# Causality

In [ ]:
# Install required packages
!pip install dowhy econml statsmodels pandas numpy scipy scikit-learn --quiet

In [ ]:
import pandas as pd
import numpy as np
from typing import List, Dict, Optional, Tuple, Union
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
from scipy import stats

try:
    import dowhy
    from dowhy import CausalModel
    DOWHY_AVAILABLE = True
    print(f"DoWhy version: {dowhy.__version__}")
except ImportError:
    DOWHY_AVAILABLE = False
    print("DoWhy not available. Install with: pip install dowhy")

# =============================================================================
# TIME-INVARIANT VARIABLE REGISTRY
# =============================================================================
TIME_INVARIANT_VARS = {
    "lat", "lon", "elevation_m", "dist_to_coast_km", "continentality_dist",
    "slope_deg", "aspect_deg", "tpi", "tri",
    "landcover_esa_2020", "landcover_igbp", "landformType",
    "permafrostType", "groundIceType",
    "soil_texture_0cm", "soil_texture_30cm",
    "soc_gkg_0cm", "soc_gkg_30cm", "soc_gkg_avg_0_30cm",
    "bulk_density_0cm", "bulk_density_30cm",
    "clay_pct_0cm", "sand_pct_0cm", "silt_pct_0cm",
}

# =============================================================================
# LITERATURE-VALIDATED DAG SPECIFICATION
# =============================================================================

def create_simplified_dag_for_treatment(treatment, available_vars=None):
    """
    Treatment-specific DAG for DoWhy (GML format).
    Confounders must cause BOTH treatment AND ALT (backdoor criterion).
    Mediators lie on the causal pathway (DO NOT adjust).
    """
    dag_specs = {
        "TDD": {
            "confounders": ["lat", "elevation_m", "dist_to_coast_km",
                            "continentality_dist", "slope_deg", "aspect_deg"],
            "mediators": ["soil_T", "soil_moisture", "NDVI_summer",
                          "LAI_summer", "NDWI_annual_mean"],
        },
        "FDD": {
            "confounders": ["lat", "elevation_m", "continentality_dist",
                            "slope_deg", "aspect_deg",
                            "wind_direction_winter", "wind_speed_winter_mean"],
            "mediators": ["soil_T_winter_mean", "soil_T_winter_min"],
        },
        "SWE_max": {
            "confounders": ["P_annual", "lat", "elevation_m",
                            "wind_direction_winter", "wind_speed_winter_mean",
                            "slope_deg", "aspect_deg", "tpi", "landformType",
                            "landcover_esa_2020", "LAI_summer", "MAAT", "FDD"],
            "mediators": ["soil_T_winter_mean", "snow_depth_winter", "snow_off_doy"],
        },
        "snow_off_doy": {
            "confounders": ["lat", "elevation_m", "MAAT",
                            "aspect_deg", "slope_deg", "SWE_max"],
            "mediators": ["TDD"],
        },
        "P_summer": {
            "confounders": ["lat", "elevation_m", "dist_to_coast_km",
                            "continentality_dist", "slope_deg", "tpi", "landformType"],
            "mediators": ["soil_moisture", "NDWI_annual_mean",
                          "NDVI_summer", "LAI_summer"],
        },
        "soil_moisture": {
            "confounders": ["tpi", "slope_deg", "landformType",
                            "soil_texture_0cm", "soc_gkg_0cm", "P_summer"],
            "mediators": ["soil_T", "NDWI_annual_mean"],
        },
        "NDWI_annual_mean": {
            "confounders": ["tpi", "slope_deg", "landformType", "soil_texture_0cm"],
            "mediators": ["soil_moisture", "soil_T"],
        },
        "FIRMS_fire_days": {
            "confounders": ["landcover_esa_2020", "T_summer_max", "heat_wave_days",
                            "dist_to_coast_km", "continentality_dist", "lat", "TDD"],
            "mediators": ["NDVI_summer", "LAI_summer", "albedo_annual", "soil_moisture"],
        },
        "heat_wave_days": {
            "confounders": ["lat", "elevation_m", "continentality_dist", "T_annual_range"],
            "mediators": ["soil_T", "soil_moisture"],
        },
        "NDVI_summer": {
            "confounders": ["lat", "MAAT", "P_summer", "soil_texture_0cm",
                            "soc_gkg_0cm", "landcover_esa_2020", "FIRMS_fire_days"],
            "mediators": ["LAI_summer", "soil_moisture", "albedo_annual"],
        },
        "ROS_days": {
            "confounders": ["P_annual", "T_winter_mean", "lat", "elevation_m",
                            "dist_to_coast_km", "continentality_dist"],
            "mediators": ["snow_depth_max", "soil_T_winter_mean"],
        },
    }

    spec = dag_specs.get(treatment,
        {"confounders": ["lat", "elevation_m"], "mediators": []})

    confounders = list(spec["confounders"])
    mediators = list(spec["mediators"])

    if available_vars is not None:
        av = set(available_vars)
        confounders = [c for c in confounders if c in av]
        mediators = [m for m in mediators if m in av]

    seen = set()
    unique = []
    for n in ["ALT", treatment] + confounders + mediators:
        if n not in seen:
            unique.append(n)
            seen.add(n)

    gml = ["graph [", "    directed 1", "",
           '    node [id "ALT" label "ALT"]',
           f'    node [id "{treatment}" label "{treatment}"]', ""]

    for c in confounders:
        gml.append(f'    node [id "{c}" label "{c}"]')
    for m in mediators:
        gml.append(f'    node [id "{m}" label "{m}"]')

    gml.append("")
    gml.append(f'    edge [source "{treatment}" target "ALT"]')

    if confounders:
        gml.append("")
        for c in confounders:
            gml.append(f'    edge [source "{c}" target "{treatment}"]')
            gml.append(f'    edge [source "{c}" target "ALT"]')

    if mediators:
        gml.append("")
        for m in mediators:
            gml.append(f'    edge [source "{treatment}" target "{m}"]')
            gml.append(f'    edge [source "{m}" target "ALT"]')

    gml.append("]")
    return "\n".join(gml)


# =============================================================================
# CAUSAL IDENTIFICATION STRATEGY
# =============================================================================

class CausalIdentificationStrategy:
    """
    Treatment-specific causal identification for ALT panel analysis.
    Site FE absorb all time-invariant confounders automatically.
    """

    ADJUSTMENT_SETS = {
        "TDD": ["lat", "elevation_m", "dist_to_coast_km", "continentality_dist",
                 "slope_deg", "aspect_deg"],
        "FDD": ["lat", "elevation_m", "continentality_dist", "slope_deg", "aspect_deg",
                 "wind_direction_winter", "wind_speed_winter_mean"],
        "SWE_max": ["P_annual", "lat", "elevation_m",
                     "wind_direction_winter", "wind_speed_winter_mean",
                     "slope_deg", "aspect_deg", "tpi", "landformType",
                     "landcover_esa_2020", "LAI_summer", "MAAT", "FDD"],
        "snow_off_doy": ["lat", "elevation_m", "MAAT", "aspect_deg", "slope_deg", "SWE_max"],
        "P_summer": ["lat", "elevation_m", "dist_to_coast_km", "continentality_dist",
                      "slope_deg", "tpi", "landformType"],
        "soil_moisture": ["tpi", "slope_deg", "landformType",
                           "soil_texture_0cm", "soc_gkg_0cm", "P_summer"],
        "NDWI_annual_mean": ["tpi", "slope_deg", "landformType", "soil_texture_0cm"],
        "FIRMS_fire_days": ["landcover_esa_2020", "T_summer_max", "heat_wave_days",
                             "dist_to_coast_km", "continentality_dist", "lat", "TDD"],
        "heat_wave_days": ["lat", "elevation_m", "continentality_dist", "T_annual_range"],
        "NDVI_summer": ["lat", "MAAT", "P_summer", "soil_texture_0cm", "soc_gkg_0cm",
                         "landcover_esa_2020", "FIRMS_fire_days"],
        "ROS_days": ["P_annual", "T_winter_mean", "lat", "elevation_m",
                      "dist_to_coast_km", "continentality_dist"],
    }

    MEDIATORS = {
        "TDD": ["soil_T", "soil_moisture", "NDVI_summer", "LAI_summer", "NDWI_annual_mean"],
        "FDD": ["soil_T_winter_mean", "soil_T_winter_min"],
        "SWE_max": ["soil_T_winter_mean", "snow_depth_winter", "snow_off_doy"],
        "snow_off_doy": ["TDD"],
        "P_summer": ["soil_moisture", "NDWI_annual_mean", "NDVI_summer", "LAI_summer"],
        "soil_moisture": ["soil_T", "NDWI_annual_mean"],
        "NDWI_annual_mean": ["soil_moisture", "soil_T"],
        "FIRMS_fire_days": ["NDVI_summer", "LAI_summer", "albedo_annual", "soil_moisture"],
        "heat_wave_days": ["soil_T", "soil_moisture"],
        "NDVI_summer": ["LAI_summer", "soil_moisture", "albedo_annual"],
        "ROS_days": ["snow_depth_max", "soil_T_winter_mean"],
    }

    EFFECT_MODIFIERS = {
        "TDD": ["permafrostType", "groundIceType", "landformType",
                 "soil_texture_0cm", "soc_gkg_0cm", "lat"],
        "FDD": ["permafrostType", "groundIceType", "landformType",
                 "continentality_dist", "soc_gkg_0cm", "snow_depth_max"],
        "SWE_max": ["permafrostType", "groundIceType", "landformType",
                     "wind_speed_winter_p90", "tpi"],
        "snow_off_doy": ["permafrostType", "groundIceType", "landformType",
                          "continentality_dist"],
        "P_summer": ["soil_texture_0cm", "soc_gkg_0cm", "slope_deg", "tpi", "landformType"],
        "soil_moisture": ["soil_texture_0cm", "soc_gkg_0cm", "slope_deg", "tpi", "landformType"],
        "NDWI_annual_mean": ["soil_texture_0cm", "slope_deg", "tpi", "landformType"],
        "FIRMS_fire_days": ["permafrostType", "groundIceType",
                             "landcover_esa_2020", "continentality_dist"],
        "heat_wave_days": ["permafrostType", "groundIceType",
                            "landcover_esa_2020", "soc_gkg_0cm", "bulk_density_0cm"],
        "NDVI_summer": ["permafrostType", "groundIceType", "landcover_esa_2020",
                         "soil_texture_0cm", "soc_gkg_0cm"],
        "ROS_days": ["permafrostType", "groundIceType",
                      "continentality_dist", "dist_to_coast_km"],
    }

    @classmethod
    def get_adjustment_set(cls, treatment, available_vars):
        base = cls.ADJUSTMENT_SETS.get(treatment, ["lat", "elevation_m"])
        meds = set(cls.MEDIATORS.get(treatment, []))
        av = set(available_vars)
        return [v for v in base if v in av and v not in meds]

    @classmethod
    def get_mediators(cls, treatment, available_vars=None):
        meds = cls.MEDIATORS.get(treatment, [])
        if available_vars is None:
            return meds
        return [m for m in meds if m in set(available_vars)]

    @classmethod
    def get_effect_modifiers(cls, treatment, available_vars):
        mods = cls.EFFECT_MODIFIERS.get(treatment, ["lat", "permafrostType"])
        return [v for v in mods if v in set(available_vars)]

    @classmethod
    def validate_no_mediator_adjustment(cls, treatment, controls):
        meds = set(cls.MEDIATORS.get(treatment, []))
        bad = [c for c in controls if c in meds]
        if bad:
            print(f"  WARNING: Controlling for mediators will bias estimate!")
            print(f"    Treatment: {treatment}, Bad controls: {bad}")
            return False
        return True


# =============================================================================
# DATA PREPARATION
# =============================================================================

def prepare_causal_data(df, outcome="Max", compute_anomalies_for_dowhy=False):
    """Prepare panel data. Anomalies only computed if needed for DoWhy."""
    data = df.copy()
    data.columns = data.columns.str.strip()

    numeric_cols = [
        "ALT", "Max", "lat", "lon", "year",
        "TDD", "FDD", "MAAT", "T_JJA_mean",
        "T_summer_max", "T_summer_mean", "T_winter_mean",
        "T_thaw_mean", "T_annual_range", "heat_wave_days",
        "P_summer", "P_annual",
        "SWE_max", "snow_off_doy", "snow_depth_max", "snow_depth_winter",
        "ROS_days", "NDVI_summer", "NDVI_max", "LAI_summer",
        "NDWI_annual_mean", "FIRMS_fire_days",
        "albedo_annual", "soil_T", "soil_moisture",
        "soil_T_winter_mean", "soil_T_winter_min",
        "elevation_m", "slope_deg", "aspect_deg", "tpi", "tri",
        "dist_to_coast_km", "continentality_dist",
        "wind_speed_winter_mean", "wind_speed_winter_max",
        "soc_gkg_0cm", "bulk_density_0cm",
    ]

    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    if "site_id" not in data.columns:
        if {"lat", "lon"}.issubset(data.columns):
            data["site_id"] = data["lat"].round(5).astype(str) + "_" + data["lon"].round(5).astype(str)
        elif "Location" in data.columns:
            data["site_id"] = data["Location"].astype(str)

    if compute_anomalies_for_dowhy and "site_id" in data.columns:
        tv = ["TDD", "FDD", "MAAT", "T_JJA_mean", "T_summer_max", "T_summer_mean",
              "T_winter_mean", "heat_wave_days", "P_summer", "P_annual",
              "SWE_max", "snow_off_doy", "snow_depth_max", "snow_depth_winter",
              "ROS_days", "NDVI_summer", "LAI_summer", "NDWI_annual_mean",
              "FIRMS_fire_days", "soil_moisture", "soil_T",
              "soil_T_winter_mean", "soil_T_winter_min", "albedo_annual"]
        for var in tv:
            if var in data.columns:
                data[f"{var}_anom"] = data[var] - data.groupby("site_id")[var].transform("mean")

    if outcome in data.columns and "site_id" in data.columns:
        sm_y = data.groupby("site_id")[outcome].transform("mean")
        data[f"{outcome}_demeaned"] = data[outcome] - sm_y
        data[f"{outcome}_site_mean"] = sm_y

    return data

# =============================================================================
# MAIN ANALYSIS CLASS
# =============================================================================

class ALTCausalAnalysis:

    def __init__(self, data, outcome='Max', compute_anomalies_for_dowhy=True, verbose=True):
        self.outcome = outcome
        self.verbose = verbose
        self.compute_anomalies_for_dowhy = compute_anomalies_for_dowhy

        self.data = prepare_causal_data(data, outcome=outcome,
                                         compute_anomalies_for_dowhy=compute_anomalies_for_dowhy)
        self.available_vars = list(self.data.columns)
        self.results = {}
        self.dowhy_models = {}

        if verbose:
            n_sites = self.data['site_id'].nunique() if 'site_id' in self.data.columns else 'N/A'
            print(f"{'='*70}")
            print(f"ALT Causal Analysis v3.4")
            print(f"{'='*70}")
            print(f"  Observations: {len(self.data)}, Sites: {n_sites}")
            if 'year' in self.data.columns:
                print(f"  Years: {int(self.data['year'].min())}-{int(self.data['year'].max())}")
            print(f"  Primary: Site FE + clustered SEs")
            print(f"  DoWhy: {DOWHY_AVAILABLE} (anomalies={compute_anomalies_for_dowhy})")

    def print_dag_assumptions(self, treatment):
        adj = CausalIdentificationStrategy.get_adjustment_set(treatment, self.available_vars)
        meds = CausalIdentificationStrategy.get_mediators(treatment, self.available_vars)
        mods = CausalIdentificationStrategy.get_effect_modifiers(treatment, self.available_vars)
        ti = [v for v in adj if v in TIME_INVARIANT_VARS]
        tv = [v for v in adj if v not in TIME_INVARIANT_VARS]

        pathways = {
            "TDD": "TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT",
            "FDD": "FDD -> (snow modulates) -> soil_T_winter -> permafrost",
            "SWE_max": "SWE -> winter insulation -> soil_T_winter -> ALT",
            "snow_off_doy": "snow_off -> thaw season -> TDD -> ALT",
            "P_summer": "P_summer -> soil_moisture -> thermal conductivity -> ALT",
            "soil_moisture": "moisture -> thermal conductivity (k) -> ALT",
            "NDWI_annual_mean": "NDWI -> heat capacity / evap cooling -> ALT",
            "FIRMS_fire_days": "fire -> veg/organic removal -> insulation -> ALT",
            "heat_wave_days": "heat extremes -> thermal pulse -> ALT",
            "NDVI_summer": "NDVI -> shading (n-factor) + ET -> GST -> ALT",
            "ROS_days": "ROS -> snowpack ice + latent heat -> winter regime",
        }

        print(f"\n{'='*60}")
        print(f"CAUSAL ASSUMPTIONS: {treatment} -> ALT")
        print(f"{'='*60}")
        print(f"  Pathway: {pathways.get(treatment, treatment + ' -> ALT')}")
        print(f"  Confounders (FE absorbed): {ti}")
        print(f"  Confounders (explicit): {tv}")
        print(f"  Mediators (no control): {meds}")
        print(f"  Effect modifiers: {mods}")

    # --- DoWhy (Secondary) ---

    def estimate_effect_dowhy(self, treatment, use_anomaly=True,
                               additional_controls=None, run_refutation=True, verbose=None):
        if not DOWHY_AVAILABLE:
            return {"error": "DoWhy not installed"}
        if verbose is None:
            verbose = self.verbose

        treat_var = f"{treatment}_anom" if (use_anomaly and f"{treatment}_anom" in self.data.columns) else treatment
        if treat_var not in self.data.columns:
            return {"error": f"Treatment {treat_var} not found"}

        adj = CausalIdentificationStrategy.get_adjustment_set(treatment, self.available_vars)
        if additional_controls:
            CausalIdentificationStrategy.validate_no_mediator_adjustment(treatment, additional_controls)
            adj = list(set(adj + additional_controls))
        adj = [v for v in adj if v in self.data.columns]

        cols = list(set([treat_var, self.outcome] + adj))
        df = self.data[cols].dropna()
        if len(df) < 50:
            return {"error": f"Insufficient data: {len(df)}"}

        if verbose:
            an = "(anomaly)" if treat_var.endswith("_anom") else "(raw)"
            print(f"\n{'='*70}")
            print(f"DOWHY: {treatment} {an} -> {self.outcome} | n={len(df)}")
            print(f"{'='*70}")

        gml = create_simplified_dag_for_treatment(treatment, self.available_vars)

        try:
            model = CausalModel(data=df, treatment=treat_var, outcome=self.outcome,
                                common_causes=adj if adj else None, graph=gml)
            self.dowhy_models[treatment] = model
            estimand = model.identify_effect(proceed_when_unidentifiable=True)

            estimates = {}

            # Linear regression
            try:
                est = model.estimate_effect(estimand, method_name="backdoor.linear_regression",
                                            confidence_intervals=True, test_significance=True)
                ci = est.get_confidence_intervals() if hasattr(est, 'get_confidence_intervals') else (np.nan, np.nan)
                estimates['linear_regression'] = {
                    'effect': float(est.value),
                    'ci_lower': float(ci[0]) if ci else np.nan,
                    'ci_upper': float(ci[1]) if ci else np.nan,
                }
                if verbose: print(f"  LR: {est.value:.4f}")
            except Exception as e:
                estimates['linear_regression'] = {'error': str(e)}

            # PS stratification
            try:
                est_ps = model.estimate_effect(estimand,
                    method_name="backdoor.propensity_score_stratification",
                    method_params={'num_strata': 5})
                estimates['propensity_score'] = {'effect': float(est_ps.value)}
                if verbose: print(f"  PS: {est_ps.value:.4f}")
            except Exception as e:
                estimates['propensity_score'] = {'error': str(e)}

            # IPW
            try:
                est_ipw = model.estimate_effect(estimand,
                    method_name="backdoor.propensity_score_weighting")
                estimates['ipw'] = {'effect': float(est_ipw.value)}
                if verbose: print(f"  IPW: {est_ipw.value:.4f}")
            except Exception as e:
                estimates['ipw'] = {'error': str(e)}

            # Refutations
            refutations = {}
            if run_refutation and 'error' not in estimates.get('linear_regression', {'error': 1}):
                est_ref = model.estimate_effect(estimand, method_name="backdoor.linear_regression")
                orig = float(est_ref.value)

                for name, method, kw, check in [
                    ("random_common_cause", "random_common_cause", {"num_simulations": 100},
                     lambda n, o: abs(n-o) < 0.1*abs(o) if o != 0 else abs(n) < 0.01),
                    ("placebo_treatment", "placebo_treatment_refuter",
                     {"placebo_type": "permute", "num_simulations": 100},
                     lambda n, o: abs(n) < 0.1*abs(o) if o != 0 else abs(n) < 0.01),
                    ("data_subset", "data_subset_refuter",
                     {"subset_fraction": 0.8, "num_simulations": 100},
                     lambda n, o: abs(n-o) < 0.2*abs(o) if o != 0 else abs(n) < 0.01),
                ]:
                    try:
                        ref = model.refute_estimate(estimand, est_ref, method_name=method, **kw)
                        new = float(ref.new_effect)
                        passed = check(new, orig)
                        refutations[name] = {'new_effect': new, 'original': orig, 'passed': passed}
                        if verbose: print(f"  {name}: {new:.4f} [{'PASS' if passed else 'FAIL'}]")
                    except Exception as e:
                        refutations[name] = {'error': str(e)}

            result = {'treatment': treatment, 'treatment_var': treat_var,
                      'outcome': self.outcome, 'confounders': adj,
                      'n_obs': len(df), 'estimates': estimates, 'refutations': refutations}

            if treatment not in self.results:
                self.results[treatment] = {}
            self.results[treatment]['dowhy'] = result
            return result

        except Exception as e:
            import traceback
            return {"error": str(e), "traceback": traceback.format_exc()}

    # --- Panel Estimation (Primary) ---

    def estimate_effect_panel(self, treatment, method="fe_cluster",
                               additional_controls=None, verbose=None):
        if verbose is None:
            verbose = self.verbose
        if treatment not in self.data.columns:
            return {"error": f"Treatment {treatment} not found"}

        adj = CausalIdentificationStrategy.get_adjustment_set(treatment, self.available_vars)
        if additional_controls:
            CausalIdentificationStrategy.validate_no_mediator_adjustment(treatment, additional_controls)
            adj = list(set(adj + additional_controls))

        cols = [treatment, self.outcome, 'site_id', 'year'] + adj
        cols = [v for v in cols if v in self.data.columns]
        df = self.data[list(set(cols))].dropna()

        if len(df) < 50:
            return {"error": f"Insufficient data: {len(df)}"}

        n_sites = df['site_id'].nunique() if 'site_id' in df.columns else 'N/A'

        if verbose:
            ti = [v for v in adj if v in TIME_INVARIANT_VARS]
            tv = [v for v in adj if v not in TIME_INVARIANT_VARS]
            print(f"\n{'='*60}")
            print(f"PANEL ({method}): {treatment} -> {self.outcome}")
            print(f"  FE-absorbed: {ti} | Explicit: {tv} | N={len(df)}, Sites={n_sites}")
            print(f"{'='*60}")

        if method == "fe_cluster":
            result = self._fe_cluster(df, treatment, adj)
        elif method == "first_diff":
            result = self._first_diff(df, treatment, adj)
        elif method == "within":
            result = self._within(df, treatment, adj)
        else:
            return {"error": f"Unknown method: {method}"}

        result.update({'treatment': treatment, 'treatment_var': treatment,
                       'adjustment_set': adj, 'method': method,
                       'n_obs': len(df), 'n_sites': n_sites})

        if treatment not in self.results:
            self.results[treatment] = {}
        self.results[treatment][method] = result

        if verbose:
            self._print(result)
        return result

    def _fe_cluster(self, df, treatment, controls):
        ca = [c for c in controls if c in df.columns]
        formula = f"{self.outcome} ~ {treatment}"
        if ca: formula += " + " + " + ".join(ca)
        formula += " + C(site_id)"
        try:
            fit = sm.OLS.from_formula(formula, df).fit(
                cov_type='cluster', cov_kwds={'groups': df['site_id']})
            # Within-site SD: matches FE identifying variation
            within_resid = df[treatment] - df.groupby('site_id')[treatment].transform('mean')
            sd_within = within_resid.std()
            effect = fit.params[treatment]
            effect_std = effect * sd_within if sd_within > 0 else np.nan
            return {'effect': effect, 'se': fit.bse[treatment],
                    'ci_lower': fit.conf_int().loc[treatment, 0],
                    'ci_upper': fit.conf_int().loc[treatment, 1],
                    'p_value': fit.pvalues[treatment], 't_stat': fit.tvalues[treatment],
                    'r_squared': fit.rsquared, 'r_squared_adj': fit.rsquared_adj,
                    'effect_std': effect_std, 'sd_treatment': sd_within}
        except Exception as e:
            return {'error': str(e)}

    def _first_diff(self, df, treatment, controls):
        df = df.sort_values(['site_id', 'year'])
        tv_ctrl = [c for c in controls if c in df.columns and c not in TIME_INVARIANT_VARS]
        dv = [v for v in [self.outcome, treatment] + tv_ctrl if v in df.columns]
        dd = df.groupby('site_id')[dv].diff().dropna()
        dd['site_id'] = df.loc[dd.index, 'site_id']
        if len(dd) < 30:
            return {'error': 'Insufficient differenced observations'}
        parts = [treatment] + [c for c in tv_ctrl if c in dd.columns]
        formula = f"{self.outcome} ~ " + " + ".join(parts)
        try:
            fit = sm.OLS.from_formula(formula, dd).fit(
                cov_type='cluster', cov_kwds={'groups': dd['site_id']})
            # SD of first differences: matches FD identifying variation
            sd_diff = dd[treatment].std()
            effect = fit.params[treatment]
            effect_std = effect * sd_diff if sd_diff > 0 else np.nan
            return {'effect': effect, 'se': fit.bse[treatment],
                    'ci_lower': fit.conf_int().loc[treatment, 0],
                    'ci_upper': fit.conf_int().loc[treatment, 1],
                    'p_value': fit.pvalues[treatment], 'r_squared': fit.rsquared,
                    'effect_std': effect_std, 'sd_treatment': sd_diff}
        except Exception as e:
            return {'error': str(e)}

    def _within(self, df, treatment, controls):
        vd = [self.outcome, treatment] + [c for c in controls if c in df.columns]
        dw = df.copy()
        for v in vd:
            if v in dw.columns:
                dw[f'{v}_w'] = dw[v] - dw.groupby('site_id')[v].transform('mean')
        ow, tw = f"{self.outcome}_w", f"{treatment}_w"
        if ow not in dw.columns or tw not in dw.columns:
            return {'error': 'Could not create within-transformed variables'}
        parts = [tw] + [f"{c}_w" for c in controls if f"{c}_w" in dw.columns]
        formula = f"{ow} ~ " + " + ".join(parts) + " - 1"
        try:
            fit = sm.OLS.from_formula(formula, dw).fit(
                cov_type='cluster', cov_kwds={'groups': dw['site_id']})
            # Within-site SD: dw[tw] is already demeaned, so its SD = within-site SD
            sd_within = dw[tw].std()
            effect = fit.params[tw]
            effect_std = effect * sd_within if sd_within > 0 else np.nan
            return {'effect': effect, 'se': fit.bse[tw],
                    'ci_lower': fit.conf_int().loc[tw, 0],
                    'ci_upper': fit.conf_int().loc[tw, 1],
                    'p_value': fit.pvalues[tw], 'r_squared': fit.rsquared,
                    'effect_std': effect_std, 'sd_treatment': sd_within}
        except Exception as e:
            return {'error': str(e)}

    def _print(self, r):
        if 'error' in r:
            print(f"  ERROR: {r['error']}"); return
        sig = "***" if r['p_value']<.001 else "**" if r['p_value']<.01 else "*" if r['p_value']<.05 else ""
        print(f"  Effect: {r['effect']:.4f} {sig}  SE: {r.get('se',np.nan):.4f}")
        print(f"  95% CI: [{r['ci_lower']:.4f}, {r['ci_upper']:.4f}]  p={r['p_value']:.4e}")
        if 'effect_std' in r and not np.isnan(r.get('effect_std', np.nan)):
            print(f"  Std. Effect: {r['effect_std']:.4f} cm/1-SD  (SD_treatment={r['sd_treatment']:.4f})")

    # --- Heterogeneity ---

    def heterogeneity_by_pretreatment(self, treatment, stratify_by, n_quantiles=3, verbose=True):
        if stratify_by not in self.data.columns:
            return {"error": f"{stratify_by} not found"}

        data = self.data.copy()
        if data[stratify_by].dtype == 'object' or data[stratify_by].nunique() <= 5:
            data['_s'] = data[stratify_by]
        else:
            try:
                data['_s'] = pd.qcut(data[stratify_by], q=n_quantiles,
                    labels=[f"Q{i+1}" for i in range(n_quantiles)], duplicates='drop')
            except ValueError:
                data['_s'] = pd.cut(data[stratify_by], bins=n_quantiles,
                    labels=[f"Q{i+1}" for i in range(n_quantiles)])

        results = {}
        for s in data['_s'].dropna().unique():
            sub = data[data['_s'] == s]
            if len(sub) < 30:
                results[str(s)] = {"error": "Insufficient data"}; continue
            t = ALTCausalAnalysis(sub, outcome=self.outcome,
                                  compute_anomalies_for_dowhy=False, verbose=False)
            r = t.estimate_effect_panel(treatment, verbose=False)
            results[str(s)] = r
            if verbose and 'effect' in r:
                sig = "*" if r['p_value'] < 0.05 else ""
                print(f"  {s}: {r['effect']:.4f} ({r['ci_lower']:.4f}, {r['ci_upper']:.4f}) {sig}  n={r['n_obs']}")

        eff = [r['effect'] for r in results.values() if 'effect' in r]
        ses = [r.get('se', 0.1) for r in results.values() if 'effect' in r]
        hr = (max(eff)-min(eff))/(2*np.mean(ses)) if len(eff)>=2 and np.mean(ses)>0 else np.nan

        return {'stratum_results': results, 'heterogeneity_ratio': hr,
                'significant_heterogeneity': hr > 1.96 if not np.isnan(hr) else None,
                'stratify_by': stratify_by}

    # --- Full Pipeline ---

    def full_analysis(self, treatments=None, methods=None,
                      run_dowhy=True, run_heterogeneity=False, verbose=True):
        if treatments is None:
            cand = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                    'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                    'heat_wave_days', 'NDVI_summer', 'ROS_days']
            treatments = [t for t in cand if t in self.data.columns]
        if methods is None:
            methods = ['fe_cluster', 'first_diff']

        all_results = {}
        for t in treatments:
            print(f"\n{'#'*70}\n# TREATMENT: {t}\n{'#'*70}")
            self.print_dag_assumptions(t)
            tr = {'panel_estimates': {}}

            for m in methods:
                tr['panel_estimates'][m] = self.estimate_effect_panel(t, method=m, verbose=verbose)

            if run_dowhy and DOWHY_AVAILABLE:
                tr['dowhy'] = self.estimate_effect_dowhy(t, verbose=verbose)

            if run_heterogeneity:
                mods = CausalIdentificationStrategy.get_effect_modifiers(t, self.available_vars)
                if mods:
                    tr['heterogeneity'] = {}
                    for mod in mods[:2]:
                        if verbose: print(f"\n--- Heterogeneity by {mod} ---")
                        tr['heterogeneity'][mod] = self.heterogeneity_by_pretreatment(
                            t, stratify_by=mod, verbose=verbose)

            all_results[t] = tr
        return all_results

    def summary_table(self):
        rows = []
        for treatment, methods in self.results.items():
            for method, result in methods.items():
                if not isinstance(result, dict) or 'error' in result:
                    continue
                if method == 'dowhy':
                    est = result.get('estimates', {}).get('linear_regression', {})
                    if 'effect' in est:
                        rows.append({'Treatment': treatment, 'Method': 'DoWhy (LR)',
                                     'Effect': est['effect'],
                                     'CI_Lower': est.get('ci_lower', np.nan),
                                     'CI_Upper': est.get('ci_upper', np.nan),
                                     'N': result.get('n_obs', '')})
                elif 'effect' in result:
                    rows.append({
                        'Treatment': treatment, 'Method': method,
                        'Effect': result['effect'], 'SE': result.get('se', np.nan),
                        'Effect_Std': result.get('effect_std', np.nan),
                        'SD_Treatment': result.get('sd_treatment', np.nan),
                        'CI_Lower': result['ci_lower'], 'CI_Upper': result['ci_upper'],
                        'p_value': result['p_value'], 'N': result.get('n_obs', ''),
                        'Sites': result.get('n_sites', ''),
                        'Sig': '***' if result['p_value']<.001 else '**' if result['p_value']<.01 else '*' if result['p_value']<.05 else ''
                    })
        return pd.DataFrame(rows)

DoWhy version: 0.14


In [ ]:
import re
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## All

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_allV3'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 2365 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 2365, Sites: 129
  Years: 1962-2024
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

PANEL (fe

  Effect: -0.0040 ***  SE: 0.0011
  95% CI: [-0.0062, -0.0018]  p=2.9522e-04
  Std. Effect: -1.5395 cm/1-SD  (SD_treatment=384.3340)

PANEL (first_diff): FDD -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg'] | Explicit: ['wind_direction_winter', 'wind_speed_winter_mean'] | N=1704, Sites=97
  Effect: -0.0016   SE: 0.0011
  95% CI: [-0.0038, 0.0005]  p=1.3227e-01
  Std. Effect: -0.8003 cm/1-SD  (SD_treatment=491.1159)

DOWHY: FDD (anomaly) -> Max | n=1704

--- Heterogeneity by permafrostType ---
  Discontinuous: -0.0049 (-0.0102, 0.0004)   n=242
  Continuous: -0.0040 (-0.0065, -0.0015) *  n=1408

--- Heterogeneity by groundIceType ---
  High: -0.0031 (-0.0068, 0.0006)   n=729


  Low: -0.0051 (-0.0100, -0.0001) *  n=415
  Medium: -0.0045 (-0.0078, -0.0012) *  n=531

######################################################################
# TREATMENT: SWE_max
######################################################################

CAUSAL ASSUMPTIONS: SWE_max -> ALT
  Pathway: SWE -> winter insulation -> soil_T_winter -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020']
  Confounders (explicit): ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD']
  Mediators (no control): ['snow_depth_winter', 'snow_off_doy']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'tpi']

PANEL (fe_cluster): SWE_max -> Max
  FE-absorbed: ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020'] | Explicit: ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD'] | N=1704, Site

  Effect: -0.0939   SE: 0.0841
  95% CI: [-0.2586, 0.0709]  p=2.6402e-01
  Std. Effect: -0.5790 cm/1-SD  (SD_treatment=6.1671)

PANEL (first_diff): snow_off_doy -> Max
  FE-absorbed: ['lat', 'elevation_m', 'aspect_deg', 'slope_deg'] | Explicit: ['MAAT', 'SWE_max'] | N=1704, Sites=97
  Effect: -0.0581   SE: 0.0573
  95% CI: [-0.1705, 0.0543]  p=3.1082e-01
  Std. Effect: -0.4785 cm/1-SD  (SD_treatment=8.2343)

DOWHY: snow_off_doy (anomaly) -> Max | n=1704

--- Heterogeneity by permafrostType ---
  Discontinuous: -0.2270 (-0.8232, 0.3692)   n=242
  Continuous: -0.0601 (-0.1898, 0.0697)   n=1408

--- Heterogeneity by groundIceType ---
  High: -0.1108 (-0.2841, 0.0625)   n=729
  Low: -0.2900 (-0.8168, 0.2367)   n=415
  Medium: 0.0212 (-0.1641, 0.2064)   n=531

######################################################################
# TREATMENT: P_summer
######################################################################

CAUSAL ASSUMPTIONS: P_summer -> ALT
  Pathway: P_summer -> soil_moist

  Effect: 0.0321 *  SE: 0.0144
  95% CI: [0.0039, 0.0602]  p=2.5610e-02
  Std. Effect: 1.3829 cm/1-SD  (SD_treatment=43.1144)

PANEL (first_diff): P_summer -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'tpi', 'landformType'] | Explicit: [] | N=1704, Sites=97
  Effect: 0.0273 **  SE: 0.0084
  95% CI: [0.0107, 0.0438]  p=1.2157e-03
  Std. Effect: 1.7229 cm/1-SD  (SD_treatment=63.1811)

DOWHY: P_summer (anomaly) -> Max | n=1704

--- Heterogeneity by soil_texture_0cm ---
  7.0: 0.0329 (-0.0001, 0.0660)   n=1457
  8.0: 0.0180 (-0.0048, 0.0407)   n=137

--- Heterogeneity by soc_gkg_0cm ---
  Q1: 0.0507 (-0.0251, 0.1264)   n=570
  Q2: 0.0309 (0.0054, 0.0564) *  n=580
  Q3: 0.0081 (-0.0080, 0.0243)   n=444

######################################################################
# TREATMENT: soil_moisture
######################################################################

CAUSAL ASSUMPTIONS: soil_moisture -> ALT
  Pathway: moisture -> th

...

  Effect: 1.0113   SE: 6.6783
  95% CI: [-12.0780, 14.1006]  p=8.7964e-01
  Std. Effect: 0.0850 cm/1-SD  (SD_treatment=0.0840)

PANEL (first_diff): NDWI_annual_mean -> Max
  FE-absorbed: ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm'] | Explicit: [] | N=2064, Sites=119
  Effect: -6.4235 *  SE: 2.8269
  95% CI: [-11.9642, -0.8828]  p=2.3073e-02
  Std. Effect: -0.7416 cm/1-SD  (SD_treatment=0.1155)

DOWHY: NDWI_annual_mean (anomaly) -> Max | n=2064

--- Heterogeneity by soil_texture_0cm ---
  7.0: -2.7164 (-14.5489, 9.1161)   n=1853
  8.0: 115.4389 (-63.8606, 294.7384)   n=186

--- Heterogeneity by slope_deg ---
  Q3: -0.4202 (-20.6969, 19.8564)   n=701
  Q2: 4.7621 (-21.6970, 31.2211)   n=765
  Q1: -0.7420 (-10.6300, 9.1460)   n=598

######################################################################
# TREATMENT: FIRMS_fire_days
######################################################################

CAUSAL ASSUMPTIONS: FIRMS_fire_days -> ALT
  Pathway: fire -> veg/organic re

  Effect: -2.5020   SE: 5.0247
  95% CI: [-12.3501, 7.3461]  p=6.1852e-01
  Std. Effect: -0.1069 cm/1-SD  (SD_treatment=0.0427)

PANEL (first_diff): FIRMS_fire_days -> Max
  FE-absorbed: ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat'] | Explicit: ['T_summer_max', 'heat_wave_days', 'TDD'] | N=1254, Sites=74
  Effect: 1.6887   SE: 1.2409
  95% CI: [-0.7435, 4.1208]  p=1.7358e-01
  Std. Effect: 0.1060 cm/1-SD  (SD_treatment=0.0628)

DOWHY: FIRMS_fire_days (anomaly) -> Max | n=1254

--- Heterogeneity by permafrostType ---
  Discontinuous: -0.0000 (-0.0000, 0.0000)   n=185
  Continuous: -2.4502 (-12.1401, 7.2397)   n=1023

--- Heterogeneity by groundIceType ---
  High: -0.0000 (-0.0000, -0.0000) *  n=560


  Low: -31.7490 (-37.5399, -25.9580) *  n=311
  Medium: 1.3088 (0.2429, 2.3747) *  n=360

######################################################################
# TREATMENT: heat_wave_days
######################################################################

CAUSAL ASSUMPTIONS: heat_wave_days -> ALT
  Pathway: heat extremes -> thermal pulse -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist']
  Confounders (explicit): ['T_annual_range']
  Mediators (no control): ['soil_T', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soc_gkg_0cm', 'bulk_density_0cm']

PANEL (fe_cluster): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist'] | Explicit: ['T_annual_range'] | N=1704, Sites=97
  Effect: 0.1929 *  SE: 0.0933
  95% CI: [0.0100, 0.3757]  p=3.8746e-02
  Std. Effect: 1.0294 cm/1-SD  (SD_treatment=5.3378)

PANEL (first_diff): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', '

  Continuous: 0.3190 (0.0748, 0.5633) *  n=1408

--- Heterogeneity by groundIceType ---
  High: 0.0870 (-0.2289, 0.4029)   n=729
  Low: 0.3241 (-0.0709, 0.7191)   n=415
  Medium: 0.2352 (0.0088, 0.4616) *  n=531

######################################################################
# TREATMENT: NDVI_summer
######################################################################

CAUSAL ASSUMPTIONS: NDVI_summer -> ALT
  Pathway: NDVI -> shading (n-factor) + ET -> GST -> ALT
  Confounders (FE absorbed): ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020']
  Confounders (explicit): ['MAAT', 'P_summer', 'FIRMS_fire_days']
  Mediators (no control): ['LAI_summer', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soil_texture_0cm', 'soc_gkg_0cm']

PANEL (fe_cluster): NDVI_summer -> Max
  FE-absorbed: ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020'] | Explicit: ['MAAT', 'P_summer', 'FIRMS_fire_days'] | N=1201, Sites=71
 

  Effect: -0.5660   SE: 0.4006
  95% CI: [-1.3512, 0.2193]  p=1.5776e-01
  Std. Effect: -0.8757 cm/1-SD  (SD_treatment=1.5473)

PANEL (first_diff): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist'] | Explicit: ['P_annual', 'T_winter_mean'] | N=1704, Sites=97
  Effect: -0.6835 *  SE: 0.3247
  95% CI: [-1.3200, -0.0471]  p=3.5295e-02
  Std. Effect: -1.4963 cm/1-SD  (SD_treatment=2.1891)

DOWHY: ROS_days (anomaly) -> Max | n=1704

--- Heterogeneity by permafrostType ---
  Discontinuous: 0.4674 (-1.1874, 2.1222)   n=242
  Continuous: -0.7418 (-1.6413, 0.1577)   n=1408

--- Heterogeneity by groundIceType ---
  High: -0.7296 (-2.1209, 0.6616)   n=729
  Low: -0.3078 (-1.8888, 1.2732)   n=415
  Medium: -0.7986 (-1.6297, 0.0326)   n=531

SUMMARY
       Treatment     Method     Effect        SE  Effect_Std  SD_Treatment    CI_Lower  CI_Upper      p_value    N  Sites Sig
             TDD fe_cluster   0.022686  0.004158    3.083235    135.906637    0.

## rapid_thickening

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_rapid_thickening.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_rapid_thickeningV3'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 341 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 341, Sites: 23
  Years: 1986-2024
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

PANEL (fe_cl

  Effect: 0.0116   SE: 0.0109
  95% CI: [-0.0097, 0.0329]  p=2.8396e-01
  Std. Effect: 2.4256 cm/1-SD  (SD_treatment=208.4319)

DOWHY: TDD (anomaly) -> Max | n=252

--- Heterogeneity by permafrostType ---
  Discontinuous: 0.0483 (-0.0347, 0.1312)   n=63
  Continuous: 0.0455 (0.0207, 0.0704) *  n=189

--- Heterogeneity by groundIceType ---
  Low: 0.0565 (0.0153, 0.0976) *  n=132
  Medium: 0.0527 (-0.0042, 0.1096)   n=62
  High: 0.0165 (-0.0028, 0.0357)   n=58

######################################################################
# TREATMENT: FDD
######################################################################

CAUSAL ASSUMPTIONS: FDD -> ALT
  Pathway: FDD -> (snow modulates) -> soil_T_winter -> permafrost
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): ['wind_direction_winter', 'wind_speed_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType'

  Effect: -0.0080 *  SE: 0.0032
  95% CI: [-0.0143, -0.0017]  p=1.2579e-02
  Std. Effect: -3.2525 cm/1-SD  (SD_treatment=406.0526)

PANEL (first_diff): FDD -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg'] | Explicit: ['wind_direction_winter', 'wind_speed_winter_mean'] | N=252, Sites=18
  Effect: 0.0009   SE: 0.0023
  95% CI: [-0.0035, 0.0054]  p=6.7803e-01
  Std. Effect: 0.5228 cm/1-SD  (SD_treatment=554.3913)

DOWHY: FDD (anomaly) -> Max | n=252

--- Heterogeneity by permafrostType ---
  Discontinuous: 0.0033 (-0.0211, 0.0278)   n=63
  Continuous: -0.0099 (-0.0159, -0.0040) *  n=189

--- Heterogeneity by groundIceType ---
  Low: -0.0073 (-0.0184, 0.0038)   n=132
  Medium: -0.0097 (-0.0247, 0.0053)   n=62


  High: -0.0074 (-0.0152, 0.0005)   n=58

######################################################################
# TREATMENT: SWE_max
######################################################################

CAUSAL ASSUMPTIONS: SWE_max -> ALT
  Pathway: SWE -> winter insulation -> soil_T_winter -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020']
  Confounders (explicit): ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD']
  Mediators (no control): ['snow_depth_winter', 'snow_off_doy']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'tpi']

PANEL (fe_cluster): SWE_max -> Max
  FE-absorbed: ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020'] | Explicit: ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD'] | N=252, Sites=18
  Effect: 12.5444   SE: 12.5549
  95% CI: [-

  Medium: -3.0691 (-30.0223, 23.8842)   n=62
  High: 66.6930 (-28.2361, 161.6221)   n=58

######################################################################
# TREATMENT: snow_off_doy
######################################################################

CAUSAL ASSUMPTIONS: snow_off_doy -> ALT
  Pathway: snow_off -> thaw season -> TDD -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'aspect_deg', 'slope_deg']
  Confounders (explicit): ['MAAT', 'SWE_max']
  Mediators (no control): ['TDD']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'continentality_dist']

PANEL (fe_cluster): snow_off_doy -> Max
  FE-absorbed: ['lat', 'elevation_m', 'aspect_deg', 'slope_deg'] | Explicit: ['MAAT', 'SWE_max'] | N=252, Sites=18
  Effect: -0.2869   SE: 0.3692
  95% CI: [-1.0105, 0.4366]  p=4.3703e-01
  Std. Effect: -1.9244 cm/1-SD  (SD_treatment=6.7067)

PANEL (first_diff): snow_off_doy -> Max
  FE-absorbed: ['lat', 'elevation_m', 'aspect_deg', 'slope_deg'] | Explici

...

  Q2: 58.4201 (-3.6022, 120.4424)   n=122

######################################################################
# TREATMENT: FIRMS_fire_days
######################################################################

CAUSAL ASSUMPTIONS: FIRMS_fire_days -> ALT
  Pathway: fire -> veg/organic removal -> insulation -> ALT
  Confounders (FE absorbed): ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat']
  Confounders (explicit): ['T_summer_max', 'heat_wave_days', 'TDD']
  Mediators (no control): ['NDVI_summer', 'LAI_summer', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'continentality_dist']

PANEL (fe_cluster): FIRMS_fire_days -> Max
  FE-absorbed: ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat'] | Explicit: ['T_summer_max', 'heat_wave_days', 'TDD'] | N=174, Sites=11
  Effect: -27.3562 ***  SE: 2.7981
  95% CI: [-32.8405, -21.8720]  p=1.4192e-22
  Std. Effect: -1.0503 cm/1-SD  (SD_treatment=0.0384)


  Continuous: -26.0924 (-32.5083, -19.6765) *  n=120

--- Heterogeneity by groundIceType ---
  Low: -28.4883 (-38.3102, -18.6664) *  n=105

######################################################################
# TREATMENT: heat_wave_days
######################################################################

CAUSAL ASSUMPTIONS: heat_wave_days -> ALT
  Pathway: heat extremes -> thermal pulse -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist']
  Confounders (explicit): ['T_annual_range']
  Mediators (no control): ['soil_T', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soc_gkg_0cm', 'bulk_density_0cm']

PANEL (fe_cluster): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist'] | Explicit: ['T_annual_range'] | N=252, Sites=18
  Effect: 0.6108 **  SE: 0.1922
  95% CI: [0.2340, 0.9876]  p=1.4875e-03
  Std. Effect: 3.9827 cm/1-SD  (SD_treatment=6.5205)

PANEL (first_diff): heat_wave_d


--- Heterogeneity by permafrostType ---
  Discontinuous: 0.0078 (-0.6370, 0.6526)   n=63
  Continuous: 0.9351 (0.3569, 1.5134) *  n=189

--- Heterogeneity by groundIceType ---
  Low: 0.6324 (0.0804, 1.1843) *  n=132
  Medium: 0.7931 (-0.9123, 2.4986)   n=62
  High: 0.4912 (-0.1138, 1.0962)   n=58

######################################################################
# TREATMENT: NDVI_summer
######################################################################

CAUSAL ASSUMPTIONS: NDVI_summer -> ALT
  Pathway: NDVI -> shading (n-factor) + ET -> GST -> ALT
  Confounders (FE absorbed): ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020']
  Confounders (explicit): ['MAAT', 'P_summer', 'FIRMS_fire_days']
  Mediators (no control): ['LAI_summer', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soil_texture_0cm', 'soc_gkg_0cm']

PANEL (fe_cluster): NDVI_summer -> Max
  FE-absorbed: ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landco


PANEL (first_diff): NDVI_summer -> Max
  FE-absorbed: ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020'] | Explicit: ['MAAT', 'P_summer', 'FIRMS_fire_days'] | N=161, Sites=10
  Effect: 56.4129   SE: 63.2602
  95% CI: [-67.5748, 180.4005]  p=3.7252e-01
  Std. Effect: 2.3880 cm/1-SD  (SD_treatment=0.0423)

DOWHY: NDVI_summer (anomaly) -> Max | n=161

--- Heterogeneity by permafrostType ---
  Discontinuous: 711.5385 (317.6089, 1105.4682) *  n=54
  Continuous: 93.3535 (5.4447, 181.2623) *  n=107

--- Heterogeneity by groundIceType ---


  Low: 436.1131 (-71.6993, 943.9254)   n=92

######################################################################
# TREATMENT: ROS_days
######################################################################

CAUSAL ASSUMPTIONS: ROS_days -> ALT
  Pathway: ROS -> snowpack ice + latent heat -> winter regime
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist']
  Confounders (explicit): ['P_annual', 'T_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'continentality_dist', 'dist_to_coast_km']

PANEL (fe_cluster): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist'] | Explicit: ['P_annual', 'T_winter_mean'] | N=252, Sites=18
  Effect: -2.0164 **  SE: 0.7663
  95% CI: [-3.5183, -0.5144]  p=8.5080e-03
  Std. Effect: -4.2577 cm/1-SD  (SD_treatment=2.1116)

PANEL (first_diff): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continent

## gradual_thickening

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_gradual_thickening.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_gradual_thickeningV3'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 787 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 787, Sites: 36
  Years: 1969-2024
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

PANEL (fe_cl

  Effect: 0.0284 ***  SE: 0.0074
  95% CI: [0.0140, 0.0428]  p=1.1180e-04
  Std. Effect: 4.7736 cm/1-SD  (SD_treatment=167.9304)

DOWHY: TDD (anomaly) -> Max | n=425

--- Heterogeneity by permafrostType ---
  Continuous: 0.0374 (0.0170, 0.0578) *  n=353

--- Heterogeneity by groundIceType ---
  High: 0.0603 (0.0237, 0.0970) *  n=132
  Low: 0.0403 (0.0218, 0.0588) *  n=135
  Medium: 0.0196 (0.0093, 0.0299) *  n=158

######################################################################
# TREATMENT: FDD
######################################################################

CAUSAL ASSUMPTIONS: FDD -> ALT
  Pathway: FDD -> (snow modulates) -> soil_T_winter -> permafrost
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): ['wind_direction_winter', 'wind_speed_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'continentality_dist', 'soc_gkg_0cm']

PANE


--- Heterogeneity by permafrostType ---
  Continuous: -0.0082 (-0.0159, -0.0005) *  n=353

--- Heterogeneity by groundIceType ---
  High: -0.0122 (-0.0337, 0.0092)   n=132
  Low: -0.0118 (-0.0191, -0.0044) *  n=135
  Medium: -0.0046 (-0.0078, -0.0013) *  n=158

######################################################################
# TREATMENT: SWE_max
######################################################################

CAUSAL ASSUMPTIONS: SWE_max -> ALT
  Pathway: SWE -> winter insulation -> soil_T_winter -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020']
  Confounders (explicit): ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD']
  Mediators (no control): ['snow_depth_winter', 'snow_off_doy']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'tpi']

PANEL (fe_cluster): SWE_max -> Max
  FE-absorbed: ['lat', 'elevation_m', 'slope_deg', 'aspe

  Effect: 3.4753   SE: 19.2081
  95% CI: [-34.1719, 41.1224]  p=8.5642e-01
  Std. Effect: 0.1501 cm/1-SD  (SD_treatment=0.0432)

PANEL (first_diff): SWE_max -> Max
  FE-absorbed: ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020'] | Explicit: ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD'] | N=425, Sites=21
  Effect: 2.8775   SE: 11.6159
  95% CI: [-19.8893, 25.6443]  p=8.0435e-01
  Std. Effect: 0.1751 cm/1-SD  (SD_treatment=0.0608)

DOWHY: SWE_max (anomaly) -> Max | n=425

--- Heterogeneity by permafrostType ---
  Continuous: 11.5713 (-48.1036, 71.2463)   n=353

--- Heterogeneity by groundIceType ---
  High: -35.0906 (-75.2674, 5.0863)   n=132
  Low: 101.0829 (70.7690, 131.3968) *  n=135
  Medium: -8.3593 (-54.4123, 37.6938)   n=158

######################################################################
# TREATMENT: snow_off_doy
######################################################################

  Effect: -0.2122 *  SE: 0.0991
  95% CI: [-0.4065, -0.0179]  p=3.2323e-02
  Std. Effect: -1.1405 cm/1-SD  (SD_treatment=5.3745)

PANEL (first_diff): snow_off_doy -> Max
  FE-absorbed: ['lat', 'elevation_m', 'aspect_deg', 'slope_deg'] | Explicit: ['MAAT', 'SWE_max'] | N=425, Sites=21
  Effect: -0.2663 **  SE: 0.0996
  95% CI: [-0.4615, -0.0711]  p=7.5021e-03
  Std. Effect: -1.8673 cm/1-SD  (SD_treatment=7.0123)

DOWHY: snow_off_doy (anomaly) -> Max | n=425

--- Heterogeneity by permafrostType ---
  Continuous: -0.1665 (-0.3780, 0.0450)   n=353

--- Heterogeneity by groundIceType ---
  High: -0.4130 (-0.8020, -0.0241) *  n=132


...

  Q1: -78.1216 (-211.8780, 55.6348)   n=141
  Q2: -124.7432 (-281.9580, 32.4715)   n=160
  Q3: -96.6073 (-181.9804, -11.2341) *  n=107

######################################################################
# TREATMENT: NDWI_annual_mean
######################################################################

CAUSAL ASSUMPTIONS: NDWI_annual_mean -> ALT
  Pathway: NDWI -> heat capacity / evap cooling -> ALT
  Confounders (FE absorbed): ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm']
  Confounders (explicit): []
  Mediators (no control): ['soil_moisture', 'soil_T']
  Effect modifiers: ['soil_texture_0cm', 'slope_deg', 'tpi', 'landformType']

PANEL (fe_cluster): NDWI_annual_mean -> Max
  FE-absorbed: ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm'] | Explicit: [] | N=689, Sites=34
  Effect: -2.6362   SE: 11.0167
  95% CI: [-24.2284, 18.9561]  p=8.1088e-01
  Std. Effect: -0.2318 cm/1-SD  (SD_treatment=0.0879)

PANEL (first_diff): NDWI_annual_mean -> Max
  FE-absorbed: ['tpi'

  Effect: 0.0000 ***  SE: 0.0000
  95% CI: [0.0000, 0.0000]  p=0.0000e+00

PANEL (first_diff): FIRMS_fire_days -> Max
  FE-absorbed: ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat'] | Explicit: ['T_summer_max', 'heat_wave_days', 'TDD'] | N=329, Sites=18
  Effect: -0.0000   SE: 0.0000
  95% CI: [-0.0000, 0.0000]  p=8.7751e-01

DOWHY: FIRMS_fire_days (anomaly) -> Max | n=329

--- Heterogeneity by permafrostType ---
  Continuous: 0.0000 (0.0000, 0.0000) *  n=262

--- Heterogeneity by groundIceType ---


  High: 0.0000 (-0.0000, 0.0000)   n=112
  Low: 0.0000 (0.0000, 0.0000) *  n=99
  Medium: -0.0000 (-0.0000, -0.0000) *  n=118

######################################################################
# TREATMENT: heat_wave_days
######################################################################

CAUSAL ASSUMPTIONS: heat_wave_days -> ALT
  Pathway: heat extremes -> thermal pulse -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist']
  Confounders (explicit): ['T_annual_range']
  Mediators (no control): ['soil_T', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soc_gkg_0cm', 'bulk_density_0cm']

PANEL (fe_cluster): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist'] | Explicit: ['T_annual_range'] | N=425, Sites=21
  Effect: 0.5246 *  SE: 0.2248
  95% CI: [0.0840, 0.9651]  p=1.9604e-02
  Std. Effect: 1.9696 cm/1-SD  (SD_treatment=3.7546)

PANEL (first_diff): heat_wave_days -> Max
  F

  Continuous: 0.6203 (0.0734, 1.1672) *  n=353

--- Heterogeneity by groundIceType ---
  High: 0.4835 (-0.4829, 1.4499)   n=132
  Low: 0.8517 (-0.1210, 1.8243)   n=135
  Medium: 0.3982 (-0.0305, 0.8269)   n=158

######################################################################
# TREATMENT: NDVI_summer
######################################################################

CAUSAL ASSUMPTIONS: NDVI_summer -> ALT
  Pathway: NDVI -> shading (n-factor) + ET -> GST -> ALT
  Confounders (FE absorbed): ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020']
  Confounders (explicit): ['MAAT', 'P_summer', 'FIRMS_fire_days']
  Mediators (no control): ['LAI_summer', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soil_texture_0cm', 'soc_gkg_0cm']

PANEL (fe_cluster): NDVI_summer -> Max
  FE-absorbed: ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020'] | Explicit: ['MAAT', 'P_summer', 'FIRMS_fire_days'] | N=312, Sites=17
  E

  High: 50.8340 (-54.7840, 156.4520)   n=95
  Low: 102.2434 (57.7930, 146.6937) *  n=99
  Medium: -2.2504 (-21.0296, 16.5289)   n=118

######################################################################
# TREATMENT: ROS_days
######################################################################

CAUSAL ASSUMPTIONS: ROS_days -> ALT
  Pathway: ROS -> snowpack ice + latent heat -> winter regime
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist']
  Confounders (explicit): ['P_annual', 'T_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'continentality_dist', 'dist_to_coast_km']

PANEL (fe_cluster): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist'] | Explicit: ['P_annual', 'T_winter_mean'] | N=425, Sites=21
  Effect: 0.7362   SE: 0.5406
  95% CI: [-0.3234, 1.7957]  p=1.7327e-01
  Std. Effect: 1.1770 cm/1-SD  (SD_treatment=1.5988)

PANEL (first_diff):

## no_trend

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_no_trend.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_no_trendV3'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 963 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 963, Sites: 51
  Years: 1962-2024
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

PANEL (fe_cl


--- Heterogeneity by permafrostType ---
  Discontinuous: -0.0037 (-0.0075, 0.0001)   n=113
  Continuous: 0.0128 (0.0060, 0.0196) *  n=649

--- Heterogeneity by groundIceType ---
  Low: 0.0023 (-0.0159, 0.0204)   n=117
  Medium: 0.0081 (0.0001, 0.0161) *  n=202


  High: 0.0133 (0.0050, 0.0217) *  n=443

######################################################################
# TREATMENT: FDD
######################################################################

CAUSAL ASSUMPTIONS: FDD -> ALT
  Pathway: FDD -> (snow modulates) -> soil_T_winter -> permafrost
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): ['wind_direction_winter', 'wind_speed_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'continentality_dist', 'soc_gkg_0cm']

PANEL (fe_cluster): FDD -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg'] | Explicit: ['wind_direction_winter', 'wind_speed_winter_mean'] | N=791, Sites=41
  Effect: -0.0015   SE: 0.0008
  95% CI: [-0.0031, 0.0001]  p=6.9127e-02
  Std. Effect: -0.5850 cm/1-SD  (SD_treatment=394.4009)

PANEL (first_diff): FDD -> Max
  FE-absorbed: ['la

  Effect: 25.9721 *  SE: 12.0727
  95% CI: [2.3100, 49.6342]  p=3.1452e-02
  Std. Effect: 1.2412 cm/1-SD  (SD_treatment=0.0478)

DOWHY: SWE_max (anomaly) -> Max | n=791

--- Heterogeneity by permafrostType ---
  Discontinuous: -49.5677 (-135.1969, 36.0615)   n=113
  Continuous: 10.7200 (-15.1888, 36.6288)   n=649

--- Heterogeneity by groundIceType ---
  Low: 12.2480 (-25.7724, 50.2683)   n=117
  Medium: -17.3313 (-32.9278, -1.7348) *  n=202
  High: 0.1035 (-54.6131, 54.8201)   n=443

######################################################################
# TREATMENT: snow_off_doy
######################################################################

CAUSAL ASSUMPTIONS: snow_off_doy -> ALT
  Pathway: snow_off -> thaw season -> TDD -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'aspect_deg', 'slope_deg']
  Confounders (explicit): ['MAAT', 'SWE_max']
  Mediators (no control): ['TDD']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'continentality_dist'

  Effect: -0.0049   SE: 0.0587
  95% CI: [-0.1198, 0.1101]  p=9.3376e-01
  Std. Effect: -0.0323 cm/1-SD  (SD_treatment=6.6163)

PANEL (first_diff): snow_off_doy -> Max
  FE-absorbed: ['lat', 'elevation_m', 'aspect_deg', 'slope_deg'] | Explicit: ['MAAT', 'SWE_max'] | N=791, Sites=41
  Effect: -0.0464   SE: 0.0665
  95% CI: [-0.1767, 0.0840]  p=4.8544e-01
  Std. Effect: -0.4099 cm/1-SD  (SD_treatment=8.8368)

DOWHY: snow_off_doy (anomaly) -> Max | n=791

--- Heterogeneity by permafrostType ---
  Discontinuous: 0.1608 (0.0878, 0.2339) *  n=113
  Continuous: -0.0351 (-0.1868, 0.1165)   n=649

--- Heterogeneity by groundIceType ---
  Low: -0.0341 (-0.5026, 0.4344)   n=117
  Medium: 0.0738 (-0.0669, 0.2145)   n=202


...


DOWHY: heat_wave_days (anomaly) -> Max | n=791

--- Heterogeneity by permafrostType ---
  Discontinuous: -0.1459 (-0.3803, 0.0885)   n=113
  Continuous: 0.0900 (-0.0857, 0.2657)   n=649

--- Heterogeneity by groundIceType ---
  Low: -0.1702 (-0.4950, 0.1545)   n=117
  Medium: 0.0391 (-0.0699, 0.1481)   n=202
  High: 0.0093 (-0.2835, 0.3020)   n=443

######################################################################
# TREATMENT: NDVI_summer
######################################################################

CAUSAL ASSUMPTIONS: NDVI_summer -> ALT
  Pathway: NDVI -> shading (n-factor) + ET -> GST -> ALT
  Confounders (FE absorbed): ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020']
  Confounders (explicit): ['MAAT', 'P_summer', 'FIRMS_fire_days']
  Mediators (no control): ['LAI_summer', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soil_texture_0cm', 'soc_gkg_0cm']

PANEL (fe_cluster): NDVI_summer -> Max
  FE-absorbe

  Effect: 36.2060 **  SE: 13.7048
  95% CI: [9.3450, 63.0670]  p=8.2457e-03
  Std. Effect: 1.4081 cm/1-SD  (SD_treatment=0.0389)

PANEL (first_diff): NDVI_summer -> Max
  FE-absorbed: ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020'] | Explicit: ['MAAT', 'P_summer', 'FIRMS_fire_days'] | N=585, Sites=34
  Effect: 29.7174 *  SE: 11.9679
  95% CI: [6.2607, 53.1741]  p=1.3025e-02
  Std. Effect: 1.5495 cm/1-SD  (SD_treatment=0.0521)

DOWHY: NDVI_summer (anomaly) -> Max | n=585

--- Heterogeneity by permafrostType ---
  Discontinuous: -45.3745 (-188.2489, 97.4998)   n=79
  Continuous: 44.0374 (18.2045, 69.8702) *  n=506

--- Heterogeneity by groundIceType ---
  Low: 77.2281 (-10.3545, 164.8107)   n=98
  Medium: -5.3816 (-37.3841, 26.6210)   n=137


  High: 40.6248 (6.0253, 75.2243) *  n=350

######################################################################
# TREATMENT: ROS_days
######################################################################

CAUSAL ASSUMPTIONS: ROS_days -> ALT
  Pathway: ROS -> snowpack ice + latent heat -> winter regime
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist']
  Confounders (explicit): ['P_annual', 'T_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'continentality_dist', 'dist_to_coast_km']

PANEL (fe_cluster): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist'] | Explicit: ['P_annual', 'T_winter_mean'] | N=791, Sites=41
  Effect: -0.1505   SE: 0.4618
  95% CI: [-1.0556, 0.7546]  p=7.4446e-01
  Std. Effect: -0.1865 cm/1-SD  (SD_treatment=1.2391)

PANEL (first_diff): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentalit

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  Medium: -0.3029 (-1.1157, 0.5099)   n=202
  High: 0.1268 (-2.2816, 2.5352)   n=443

SUMMARY
       Treatment     Method     Effect        SE  Effect_Std  SD_Treatment   CI_Lower  CI_Upper      p_value   N  Sites Sig
             TDD fe_cluster   0.009729  0.002836    1.346668    138.413852   0.004171  0.015288 6.022036e-04 791     41 ***
             TDD first_diff   0.007213  0.004523    1.308826    181.452859  -0.001651  0.016077 1.107501e-01 791     41    
             FDD fe_cluster  -0.001483  0.000816   -0.584960    394.400877  -0.003082  0.000116 6.912675e-02 791     41    
             FDD first_diff  -0.000339  0.000948   -0.164228    484.383801  -0.002197  0.001519 7.205931e-01 791     41    
         SWE_max fe_cluster   1.791559 13.630004    0.064168      0.035817 -24.922758 28.505877 8.954254e-01 791     41    
         SWE_max first_diff  25.972105 12.072708    1.241172      0.047789   2.310032 49.634179 3.145196e-02 791     41   *
    snow_off_doy fe_cluster  -0.004875

## gradual_thinning

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_gradual_thinning.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_gradual_thinningV3'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 189 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 189, Sites: 14
  Years: 1986-2023
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

PANEL (fe_cl

  Effect: 0.0013   SE: 0.0052
  95% CI: [-0.0089, 0.0114]  p=8.0925e-01
  Std. Effect: 0.4305 cm/1-SD  (SD_treatment=343.2723)

PANEL (first_diff): FDD -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg'] | Explicit: ['wind_direction_winter', 'wind_speed_winter_mean'] | N=151, Sites=12
  Effect: -0.0023   SE: 0.0043
  95% CI: [-0.0106, 0.0061]  p=5.9500e-01
  Std. Effect: -1.0787 cm/1-SD  (SD_treatment=476.3875)

DOWHY: FDD (anomaly) -> Max | n=151

--- Heterogeneity by permafrostType ---
  Continuous: 0.0051 (-0.0026, 0.0127)   n=132

--- Heterogeneity by groundIceType ---
  Medium: -0.0046 (-0.0300, 0.0208)   n=53


  High: 0.0035 (-0.0081, 0.0151)   n=87

######################################################################
# TREATMENT: SWE_max
######################################################################

CAUSAL ASSUMPTIONS: SWE_max -> ALT
  Pathway: SWE -> winter insulation -> soil_T_winter -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020']
  Confounders (explicit): ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD']
  Mediators (no control): ['snow_depth_winter', 'snow_off_doy']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'tpi']

PANEL (fe_cluster): SWE_max -> Max
  FE-absorbed: ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020'] | Explicit: ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD'] | N=151, Sites=12
  Effect: 22.6731   SE: 37.5982
  95% CI: [-5

  High: 49.3350 (-25.6110, 124.2811)   n=87

######################################################################
# TREATMENT: snow_off_doy
######################################################################

CAUSAL ASSUMPTIONS: snow_off_doy -> ALT
  Pathway: snow_off -> thaw season -> TDD -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'aspect_deg', 'slope_deg']
  Confounders (explicit): ['MAAT', 'SWE_max']
  Mediators (no control): ['TDD']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'continentality_dist']

PANEL (fe_cluster): snow_off_doy -> Max
  FE-absorbed: ['lat', 'elevation_m', 'aspect_deg', 'slope_deg'] | Explicit: ['MAAT', 'SWE_max'] | N=151, Sites=12
  Effect: 0.2159   SE: 0.3685
  95% CI: [-0.5063, 0.9381]  p=5.5795e-01
  Std. Effect: 1.1127 cm/1-SD  (SD_treatment=5.1539)

PANEL (first_diff): snow_off_doy -> Max
  FE-absorbed: ['lat', 'elevation_m', 'aspect_deg', 'slope_deg'] | Explicit: ['MAAT', 'SWE_max'] | N=151, Sites=12
  Effe

  High: 0.1037 (-0.4303, 0.6377)   n=87

######################################################################
# TREATMENT: P_summer
######################################################################

CAUSAL ASSUMPTIONS: P_summer -> ALT
  Pathway: P_summer -> soil_moisture -> thermal conductivity -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'tpi', 'landformType']
  Confounders (explicit): []
  Mediators (no control): ['soil_moisture', 'NDWI_annual_mean', 'NDVI_summer', 'LAI_summer']
  Effect modifiers: ['soil_texture_0cm', 'soc_gkg_0cm', 'slope_deg', 'tpi', 'landformType']

PANEL (fe_cluster): P_summer -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'tpi', 'landformType'] | Explicit: [] | N=151, Sites=12
  Effect: 0.0218   SE: 0.0316
  95% CI: [-0.0401, 0.0837]  p=4.9047e-01
  Std. Effect: 0.7237 cm/1-SD  (SD_treatment=33.2188)

PANEL (first_diff): P_summer ->

...


PANEL (fe_cluster): NDWI_annual_mean -> Max
  FE-absorbed: ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm'] | Explicit: [] | N=153, Sites=12
  Effect: -25.7134 ***  SE: 5.1576
  95% CI: [-35.8220, -15.6047]  p=6.1780e-07
  Std. Effect: -2.6503 cm/1-SD  (SD_treatment=0.1031)

PANEL (first_diff): NDWI_annual_mean -> Max
  FE-absorbed: ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm'] | Explicit: [] | N=153, Sites=12
  Effect: -6.1045   SE: 3.9009
  95% CI: [-13.7502, 1.5411]  p=1.1761e-01
  Std. Effect: -0.8711 cm/1-SD  (SD_treatment=0.1427)

DOWHY: NDWI_annual_mean (anomaly) -> Max | n=153

--- Heterogeneity by soil_texture_0cm ---
  7.0: -25.7134 (-35.8220, -15.6047) *  n=153

--- Heterogeneity by slope_deg ---


  Q2: -25.9120 (-57.4730, 5.6490)   n=53
  Q3: -29.8252 (-43.8042, -15.8461) *  n=59

######################################################################
# TREATMENT: FIRMS_fire_days
######################################################################

CAUSAL ASSUMPTIONS: FIRMS_fire_days -> ALT
  Pathway: fire -> veg/organic removal -> insulation -> ALT
  Confounders (FE absorbed): ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat']
  Confounders (explicit): ['T_summer_max', 'heat_wave_days', 'TDD']
  Mediators (no control): ['NDVI_summer', 'LAI_summer', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'continentality_dist']

PANEL (fe_cluster): FIRMS_fire_days -> Max
  FE-absorbed: ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat'] | Explicit: ['T_summer_max', 'heat_wave_days', 'TDD'] | N=82, Sites=6
  Effect: -0.0000   SE: 0.0000
  95% CI: [-0.0000, 0.0000]  p=9.0091e-01

PANEL (first_diff):


--- Heterogeneity by groundIceType ---
  High: 0.0000 (0.0000, 0.0000) *  n=60

######################################################################
# TREATMENT: heat_wave_days
######################################################################

CAUSAL ASSUMPTIONS: heat_wave_days -> ALT
  Pathway: heat extremes -> thermal pulse -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist']
  Confounders (explicit): ['T_annual_range']
  Mediators (no control): ['soil_T', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soc_gkg_0cm', 'bulk_density_0cm']

PANEL (fe_cluster): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist'] | Explicit: ['T_annual_range'] | N=151, Sites=12
  Effect: -0.1935   SE: 0.3156
  95% CI: [-0.8120, 0.4250]  p=5.3975e-01
  Std. Effect: -1.0564 cm/1-SD  (SD_treatment=5.4592)

PANEL (first_diff): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continen

  Continuous: -0.4652 (-1.0890, 0.1586)   n=132

--- Heterogeneity by groundIceType ---
  Medium: 0.2484 (-0.2639, 0.7606)   n=53
  High: -0.4834 (-1.2025, 0.2358)   n=87

######################################################################
# TREATMENT: NDVI_summer
######################################################################

CAUSAL ASSUMPTIONS: NDVI_summer -> ALT
  Pathway: NDVI -> shading (n-factor) + ET -> GST -> ALT
  Confounders (FE absorbed): ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020']
  Confounders (explicit): ['MAAT', 'P_summer', 'FIRMS_fire_days']
  Mediators (no control): ['LAI_summer', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soil_texture_0cm', 'soc_gkg_0cm']

PANEL (fe_cluster): NDVI_summer -> Max
  FE-absorbed: ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020'] | Explicit: ['MAAT', 'P_summer', 'FIRMS_fire_days'] | N=82, Sites=6
  Effect: 72.6626   SE: 44.6597
  95% CI: [-1

  Effect: -1.7887 *  SE: 0.8522
  95% CI: [-3.4590, -0.1184]  p=3.5823e-02
  Std. Effect: -4.9413 cm/1-SD  (SD_treatment=2.7625)

DOWHY: ROS_days (anomaly) -> Max | n=151

--- Heterogeneity by permafrostType ---
  Continuous: -1.3506 (-3.1577, 0.4564)   n=132

--- Heterogeneity by groundIceType ---
  Medium: -2.7592 (-5.7282, 0.2097)   n=53
  High: -0.2684 (-3.4741, 2.9372)   n=87

SUMMARY
       Treatment     Method        Effect           SE  Effect_Std  SD_Treatment      CI_Lower      CI_Upper      p_value   N  Sites Sig
             TDD fe_cluster -2.985374e-03 1.037660e-02   -0.362896    121.557871 -2.332314e-02  1.735239e-02 7.735745e-01 151     12    
             TDD first_diff  8.885139e-03 5.465728e-03    1.453707    163.611114 -1.827491e-03  1.959777e-02 1.040328e-01 151     12    
             FDD fe_cluster  1.254181e-03 5.195475e-03    0.430525    343.272301 -8.928763e-03  1.143712e-02 8.092462e-01 151     12    
             FDD first_diff -2.264390e-03 4.259538e-03   -1

## rapid_thinning

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_rapid_thinning.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_rapid_thinningV3'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 29 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 29, Sites: 3
  Years: 1986-2018
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

--- Heterogenei

## transitional

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_transitional.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_transitionalV3'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*?)V3', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 56 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 56, Sites: 2
  Years: 1993-2023
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

PANEL (fe_clust

  Medium: 0.0313 (-0.0102, 0.0728)   n=56

######################################################################
# TREATMENT: FDD
######################################################################

CAUSAL ASSUMPTIONS: FDD -> ALT
  Pathway: FDD -> (snow modulates) -> soil_T_winter -> permafrost
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): ['wind_direction_winter', 'wind_speed_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'continentality_dist', 'soc_gkg_0cm']

PANEL (fe_cluster): FDD -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg'] | Explicit: ['wind_direction_winter', 'wind_speed_winter_mean'] | N=56, Sites=2
  Effect: -0.0079   SE: 0.0048
  95% CI: [-0.0172, 0.0015]  p=9.9616e-02
  Std. Effect: -4.0915 cm/1-SD  (SD_treatment=520.7927)

PANEL (first_diff): FDD -> Max
  FE-absorbed: ['lat

  Effect: 6.0895   SE: 34.9760
  95% CI: [-62.4623, 74.6413]  p=8.6178e-01
  Std. Effect: 0.1809 cm/1-SD  (SD_treatment=0.0297)

PANEL (first_diff): SWE_max -> Max
  FE-absorbed: ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020'] | Explicit: ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD'] | N=56, Sites=2
  Effect: -46.3421   SE: 45.8986
  95% CI: [-136.3017, 43.6176]  p=3.1266e-01
  Std. Effect: -1.6826 cm/1-SD  (SD_treatment=0.0363)

DOWHY: SWE_max (anomaly) -> Max | n=56

--- Heterogeneity by permafrostType ---
  Continuous: 6.0895 (-62.4623, 74.6413)   n=56

--- Heterogeneity by groundIceType ---


  Medium: 6.0895 (-62.4623, 74.6413)   n=56

######################################################################
# TREATMENT: snow_off_doy
######################################################################

CAUSAL ASSUMPTIONS: snow_off_doy -> ALT
  Pathway: snow_off -> thaw season -> TDD -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'aspect_deg', 'slope_deg']
  Confounders (explicit): ['MAAT', 'SWE_max']
  Mediators (no control): ['TDD']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'continentality_dist']

PANEL (fe_cluster): snow_off_doy -> Max
  FE-absorbed: ['lat', 'elevation_m', 'aspect_deg', 'slope_deg'] | Explicit: ['MAAT', 'SWE_max'] | N=56, Sites=2
  Effect: -0.2633   SE: 0.2567
  95% CI: [-0.7665, 0.2398]  p=3.0503e-01
  Std. Effect: -1.8176 cm/1-SD  (SD_treatment=6.9028)

PANEL (first_diff): snow_off_doy -> Max
  FE-absorbed: ['lat', 'elevation_m', 'aspect_deg', 'slope_deg'] | Explicit: ['MAAT', 'SWE_max'] | N=56, Sites=2
  Effect

  Effect: 0.0260 ***  SE: 0.0058
  95% CI: [0.0146, 0.0374]  p=8.0182e-06
  Std. Effect: 1.3316 cm/1-SD  (SD_treatment=51.2414)

PANEL (first_diff): P_summer -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'tpi', 'landformType'] | Explicit: [] | N=56, Sites=2
  Effect: 0.0288 ***  SE: 0.0028
  95% CI: [0.0233, 0.0343]  p=9.1601e-25
  Std. Effect: 1.9796 cm/1-SD  (SD_treatment=68.6308)

DOWHY: P_summer (anomaly) -> Max | n=56

--- Heterogeneity by soil_texture_0cm ---
  7: 0.0260 (0.0146, 0.0374) *  n=56

--- Heterogeneity by soc_gkg_0cm ---

######################################################################
# TREATMENT: soil_moisture
######################################################################

CAUSAL ASSUMPTIONS: soil_moisture -> ALT
  Pathway: moisture -> thermal conductivity (k) -> ALT
  Confounders (FE absorbed): ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm']
  Confounders (explicit): ['P_su

  Effect: -48.3127   SE: 112.1120
  95% CI: [-268.0481, 171.4227]  p=6.6652e-01
  Std. Effect: -1.1906 cm/1-SD  (SD_treatment=0.0246)

DOWHY: soil_moisture (anomaly) -> Max | n=56

--- Heterogeneity by soil_texture_0cm ---
  7: -182.9199 (-802.7769, 436.9371)   n=56

--- Heterogeneity by soc_gkg_0cm ---

######################################################################
# TREATMENT: NDWI_annual_mean
######################################################################

CAUSAL ASSUMPTIONS: NDWI_annual_mean -> ALT
  Pathway: NDWI -> heat capacity / evap cooling -> ALT
  Confounders (FE absorbed): ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm']
  Confounders (explicit): []
  Mediators (no control): ['soil_moisture', 'soil_T']
  Effect modifiers: ['soil_texture_0cm', 'slope_deg', 'tpi', 'landformType']

PANEL (fe_cluster): NDWI_annual_mean -> Max
  FE-absorbed: ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm'] | Explicit: [] | N=54, Sites=2
  Effect: 12.5054 ***  SE: 2

  Continuous: 0.6836 (-0.3820, 1.7491)   n=56

--- Heterogeneity by groundIceType ---
  Medium: 0.6836 (-0.3820, 1.7491)   n=56

######################################################################
# TREATMENT: NDVI_summer
######################################################################

CAUSAL ASSUMPTIONS: NDVI_summer -> ALT
  Pathway: NDVI -> shading (n-factor) + ET -> GST -> ALT
  Confounders (FE absorbed): ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020']
  Confounders (explicit): ['MAAT', 'P_summer', 'FIRMS_fire_days']
  Mediators (no control): ['LAI_summer', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soil_texture_0cm', 'soc_gkg_0cm']

--- Heterogeneity by permafrostType ---

--- Heterogeneity by groundIceType ---

######################################################################
# TREATMENT: ROS_days
######################################################################

CAUSAL ASSUMPTIONS: ROS_days 

# Permafrost Zonation Causality

ContinuousPermafrost

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================
import re
import os

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_ContinuousPermafrost.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_ContinuousPermafrost'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*)', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*)', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 1921 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 1921, Sites: 104
  Years: 1962-2024
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

PANEL (fe

  Effect: 0.0244 ***  SE: 0.0044
  95% CI: [0.0158, 0.0330]  p=2.8631e-08
  Std. Effect: 3.2139 cm/1-SD  (SD_treatment=131.7532)

PANEL (first_diff): TDD -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg'] | Explicit: [] | N=1408, Sites=78
  Effect: 0.0150 ***  SE: 0.0041
  95% CI: [0.0069, 0.0231]  p=2.9235e-04
  Std. Effect: 2.6421 cm/1-SD  (SD_treatment=176.3556)

DOWHY: TDD (anomaly) -> Max | n=1408

--- Heterogeneity by permafrostType ---
  Continuous: 0.0244 (0.0158, 0.0330) *  n=1408

--- Heterogeneity by groundIceType ---
  Low: 0.0279 (0.0081, 0.0478) *  n=343
  Medium: 0.0266 (0.0132, 0.0401) *  n=387
  High: 0.0211 (0.0081, 0.0342) *  n=678

######################################################################
# TREATMENT: FDD
######################################################################

CAUSAL ASSUMPTIONS: FDD -> ALT
  Pathway: FDD -> (snow modulates) -> soil_T_winter -> permafrost
  Confounders (FE a

  Effect: -0.0040 **  SE: 0.0013
  95% CI: [-0.0065, -0.0015]  p=1.4499e-03
  Std. Effect: -1.5432 cm/1-SD  (SD_treatment=385.1795)

PANEL (first_diff): FDD -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg'] | Explicit: ['wind_direction_winter', 'wind_speed_winter_mean'] | N=1408, Sites=78
  Effect: -0.0016   SE: 0.0013
  95% CI: [-0.0041, 0.0008]  p=1.9647e-01
  Std. Effect: -0.7998 cm/1-SD  (SD_treatment=490.3082)

DOWHY: FDD (anomaly) -> Max | n=1408

--- Heterogeneity by permafrostType ---
  Continuous: -0.0040 (-0.0065, -0.0015) *  n=1408

--- Heterogeneity by groundIceType ---
  Low: -0.0066 (-0.0113, -0.0019) *  n=343
  Medium: -0.0037 (-0.0077, 0.0004)   n=387
  High: -0.0027 (-0.0068, 0.0013)   n=678

######################################################################
# TREATMENT: SWE_max
######################################################################

CAUSAL ASSUMPTIONS: SWE_max -> ALT
  Pathway: SWE -> winter insulation -

  Effect: 21.6366 **  SE: 7.7426
  95% CI: [6.4614, 36.8117]  p=5.1980e-03
  Std. Effect: 1.2808 cm/1-SD  (SD_treatment=0.0592)

DOWHY: SWE_max (anomaly) -> Max | n=1408

--- Heterogeneity by permafrostType ---
  Continuous: 11.3769 (-7.0308, 29.7846)   n=1408

--- Heterogeneity by groundIceType ---
  Low: 41.8766 (12.8183, 70.9350) *  n=343
  Medium: -14.0532 (-27.2216, -0.8848) *  n=387
  High: 17.7777 (-19.8267, 55.3821)   n=678

######################################################################
# TREATMENT: snow_off_doy
######################################################################

CAUSAL ASSUMPTIONS: snow_off_doy -> ALT
  Pathway: snow_off -> thaw season -> TDD -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'aspect_deg', 'slope_deg']
  Confounders (explicit): ['MAAT', 'SWE_max']
  Mediators (no control): ['TDD']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'continentality_dist']

PANEL (fe_cluster): snow_off_doy -> Max
  FE-absor

  Effect: -0.0210   SE: 0.0621
  95% CI: [-0.1428, 0.1007]  p=7.3494e-01
  Std. Effect: -0.1565 cm/1-SD  (SD_treatment=7.4419)

DOWHY: snow_off_doy (anomaly) -> Max | n=1408

--- Heterogeneity by permafrostType ---
  Continuous: -0.0601 (-0.1898, 0.0697)   n=1408

--- Heterogeneity by groundIceType ---
  Low: -0.0198 (-0.3097, 0.2701)   n=343
  Medium: -0.1436 (-0.3226, 0.0354)   n=387


...

  Q2: -71.8901 (-150.9701, 7.1899)   n=480
  Q3: -83.6185 (-131.4044, -35.8326) *  n=352

######################################################################
# TREATMENT: NDWI_annual_mean
######################################################################

CAUSAL ASSUMPTIONS: NDWI_annual_mean -> ALT
  Pathway: NDWI -> heat capacity / evap cooling -> ALT
  Confounders (FE absorbed): ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm']
  Confounders (explicit): []
  Mediators (no control): ['soil_moisture', 'soil_T']
  Effect modifiers: ['soil_texture_0cm', 'slope_deg', 'tpi', 'landformType']

PANEL (fe_cluster): NDWI_annual_mean -> Max
  FE-absorbed: ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm'] | Explicit: [] | N=1695, Sites=95
  Effect: -8.4555   SE: 5508.5798
  95% CI: [-10805.0734, 10788.1625]  p=9.9878e-01
  Std. Effect: -0.6982 cm/1-SD  (SD_treatment=0.0826)

PANEL (first_diff): NDWI_annual_mean -> Max
  FE-absorbed: ['tpi', 'slope_deg', 'landformType', 'soil_

  Effect: -2.4502   SE: 4.9439
  95% CI: [-12.1401, 7.2397]  p=6.2018e-01
  Std. Effect: -0.1159 cm/1-SD  (SD_treatment=0.0473)

PANEL (first_diff): FIRMS_fire_days -> Max
  FE-absorbed: ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat'] | Explicit: ['T_summer_max', 'heat_wave_days', 'TDD'] | N=1023, Sites=57
  Effect: 1.5307   SE: 1.2979
  95% CI: [-1.0132, 4.0746]  p=2.3826e-01
  Std. Effect: 0.1062 cm/1-SD  (SD_treatment=0.0694)

DOWHY: FIRMS_fire_days (anomaly) -> Max | n=1023

--- Heterogeneity by permafrostType ---
  Continuous: -2.4502 (-12.1401, 7.2397)   n=1023

--- Heterogeneity by groundIceType ---
  Low: -30.7335 (-38.2382, -23.2288) *  n=239


  Medium: 1.3964 (-0.3862, 3.1790)   n=269
  High: -0.0000 (-0.0000, -0.0000) *  n=515

######################################################################
# TREATMENT: heat_wave_days
######################################################################

CAUSAL ASSUMPTIONS: heat_wave_days -> ALT
  Pathway: heat extremes -> thermal pulse -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist']
  Confounders (explicit): ['T_annual_range']
  Mediators (no control): ['soil_T', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soc_gkg_0cm', 'bulk_density_0cm']

PANEL (fe_cluster): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist'] | Explicit: ['T_annual_range'] | N=1408, Sites=78
  Effect: 0.3190 *  SE: 0.1246
  95% CI: [0.0748, 0.5633]  p=1.0468e-02
  Std. Effect: 1.4733 cm/1-SD  (SD_treatment=4.6184)

PANEL (first_diff): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'co


--- Heterogeneity by permafrostType ---
  Continuous: 47.5106 (22.4277, 72.5935) *  n=993

--- Heterogeneity by groundIceType ---
  Low: 74.5103 (32.1954, 116.8252) *  n=226
  Medium: 23.9820 (-18.2853, 66.2493)   n=269


  High: 50.6495 (19.6068, 81.6921) *  n=498

######################################################################
# TREATMENT: ROS_days
######################################################################

CAUSAL ASSUMPTIONS: ROS_days -> ALT
  Pathway: ROS -> snowpack ice + latent heat -> winter regime
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist']
  Confounders (explicit): ['P_annual', 'T_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'continentality_dist', 'dist_to_coast_km']

PANEL (fe_cluster): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist'] | Explicit: ['P_annual', 'T_winter_mean'] | N=1408, Sites=78
  Effect: -0.7418   SE: 0.4589
  95% CI: [-1.6413, 0.1577]  p=1.0604e-01
  Std. Effect: -1.1360 cm/1-SD  (SD_treatment=1.5315)

PANEL (first_diff): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continental

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================
import re
import os

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_nonContinuousPermafrost.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_nonContinuousPermafrost'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*)', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*)', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 444 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 444, Sites: 25
  Years: 1969-2024
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

PANEL (fe_cl

  Effect: 0.0168   SE: 0.0102
  95% CI: [-0.0033, 0.0368]  p=1.0203e-01
  Std. Effect: 2.5873 cm/1-SD  (SD_treatment=154.3874)

PANEL (first_diff): TDD -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg'] | Explicit: [] | N=296, Sites=19
  Effect: 0.0066   SE: 0.0081
  95% CI: [-0.0093, 0.0225]  p=4.1526e-01
  Std. Effect: 1.3307 cm/1-SD  (SD_treatment=201.7418)

DOWHY: TDD (anomaly) -> Max | n=296

--- Heterogeneity by permafrostType ---
  Discontinuous: 0.0138 (-0.0105, 0.0381)   n=242

--- Heterogeneity by groundIceType ---
  Low: 0.0541 (-0.0273, 0.1356)   n=72
  Medium: 0.0023 (-0.0112, 0.0158)   n=144
  High: 0.0192 (-0.0544, 0.0929)   n=51

######################################################################
# TREATMENT: FDD
######################################################################

CAUSAL ASSUMPTIONS: FDD -> ALT
  Pathway: FDD -> (snow modulates) -> soil_T_winter -> permafrost
  Confounders (FE absorbe

  Effect: -0.0025   SE: 0.0018
  95% CI: [-0.0061, 0.0011]  p=1.7514e-01
  Std. Effect: -1.2291 cm/1-SD  (SD_treatment=495.8495)

DOWHY: FDD (anomaly) -> Max | n=296

--- Heterogeneity by permafrostType ---
  Discontinuous: -0.0049 (-0.0102, 0.0004)   n=242

--- Heterogeneity by groundIceType ---
  Low: 0.0067 (-0.0242, 0.0375)   n=72
  Medium: -0.0072 (-0.0139, -0.0005) *  n=144
  High: -0.0070 (-0.0094, -0.0047) *  n=51

######################################################################
# TREATMENT: SWE_max
######################################################################

CAUSAL ASSUMPTIONS: SWE_max -> ALT
  Pathway: SWE -> winter insulation -> soil_T_winter -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020']
  Confounders (explicit): ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD']
  Mediators (no control): ['snow_depth_winter', 'snow_off_doy']
  Effec

  Effect: -17.8860   SE: 26.2691
  95% CI: [-69.3725, 33.6004]  p=4.9595e-01
  Std. Effect: -0.8756 cm/1-SD  (SD_treatment=0.0490)

PANEL (first_diff): SWE_max -> Max
  FE-absorbed: ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020'] | Explicit: ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD'] | N=296, Sites=19
  Effect: -24.1845   SE: 20.1402
  95% CI: [-63.6585, 15.2896]  p=2.2983e-01
  Std. Effect: -1.6814 cm/1-SD  (SD_treatment=0.0695)

DOWHY: SWE_max (anomaly) -> Max | n=296

--- Heterogeneity by permafrostType ---
  Discontinuous: -12.9187 (-97.3934, 71.5560)   n=242

--- Heterogeneity by groundIceType ---
  Low: 88.7411 (-64.6685, 242.1508)   n=72
  Medium: 34.8293 (-31.5854, 101.2439)   n=144
  High: -48.0677 (-140.8070, 44.6716)   n=51

######################################################################
# TREATMENT: snow_off_doy
############################################################

  Effect: -0.1149   SE: 0.1158
  95% CI: [-0.3419, 0.1120]  p=3.2095e-01
  Std. Effect: -1.2994 cm/1-SD  (SD_treatment=11.3071)

DOWHY: snow_off_doy (anomaly) -> Max | n=296

--- Heterogeneity by permafrostType ---
  Discontinuous: -0.2270 (-0.8232, 0.3692)   n=242

--- Heterogeneity by groundIceType ---
  Low: -1.1332 (-2.7195, 0.4531)   n=72
  Medium: 0.1824 (-0.2105, 0.5754)   n=144


...

  Q1: -379.7091 (-1062.7407, 303.3225)   n=116
  Q3: 147.7597 (15.7550, 279.7644) *  n=123

######################################################################
# TREATMENT: NDWI_annual_mean
######################################################################

CAUSAL ASSUMPTIONS: NDWI_annual_mean -> ALT
  Pathway: NDWI -> heat capacity / evap cooling -> ALT
  Confounders (FE absorbed): ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm']
  Confounders (explicit): []
  Mediators (no control): ['soil_moisture', 'soil_T']
  Effect modifiers: ['soil_texture_0cm', 'slope_deg', 'tpi', 'landformType']

PANEL (fe_cluster): NDWI_annual_mean -> Max
  FE-absorbed: ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm'] | Explicit: [] | N=369, Sites=24
  Effect: 37.3536   SE: 39.3826
  95% CI: [-39.8349, 114.5420]  p=3.4289e-01
  Std. Effect: 3.3770 cm/1-SD  (SD_treatment=0.0904)

PANEL (first_diff): NDWI_annual_mean -> Max
  FE-absorbed: ['tpi', 'slope_deg', 'landformType', 'soil_texture

  Q3: 6.4788 (-68.8970, 81.8545)   n=106

######################################################################
# TREATMENT: FIRMS_fire_days
######################################################################

CAUSAL ASSUMPTIONS: FIRMS_fire_days -> ALT
  Pathway: fire -> veg/organic removal -> insulation -> ALT
  Confounders (FE absorbed): ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat']
  Confounders (explicit): ['T_summer_max', 'heat_wave_days', 'TDD']
  Mediators (no control): ['NDVI_summer', 'LAI_summer', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'continentality_dist']

PANEL (fe_cluster): FIRMS_fire_days -> Max
  FE-absorbed: ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat'] | Explicit: ['T_summer_max', 'heat_wave_days', 'TDD'] | N=231, Sites=17
  Effect: 0.0000 ***  SE: 0.0000
  95% CI: [0.0000, 0.0000]  p=0.0000e+00

PANEL (first_diff): FIRMS_fire_days -> Max
  FE-absorbed: ['

  Low: 0.0000 (0.0000, 0.0000) *  n=72
  Medium: 0.0000 (0.0000, 0.0000) *  n=91

######################################################################
# TREATMENT: heat_wave_days
######################################################################

CAUSAL ASSUMPTIONS: heat_wave_days -> ALT
  Pathway: heat extremes -> thermal pulse -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist']
  Confounders (explicit): ['T_annual_range']
  Mediators (no control): ['soil_T', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soc_gkg_0cm', 'bulk_density_0cm']

PANEL (fe_cluster): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist'] | Explicit: ['T_annual_range'] | N=296, Sites=19
  Effect: -0.0198   SE: 0.1128
  95% CI: [-0.2409, 0.2014]  p=8.6101e-01
  Std. Effect: -0.1565 cm/1-SD  (SD_treatment=7.9215)

PANEL (first_diff): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'contine

  Effect: 21.7147   SE: 48.8258
  95% CI: [-73.9822, 117.4115]  p=6.5651e-01
  Std. Effect: 0.9086 cm/1-SD  (SD_treatment=0.0418)

DOWHY: NDVI_summer (anomaly) -> Max | n=208

--- Heterogeneity by permafrostType ---
  Discontinuous: 174.7752 (-158.4559, 508.0064)   n=185

--- Heterogeneity by groundIceType ---
  Low: 619.3524 (217.9333, 1020.7715) *  n=72


  Medium: -8.4866 (-56.2731, 39.3000)   n=91

######################################################################
# TREATMENT: ROS_days
######################################################################

CAUSAL ASSUMPTIONS: ROS_days -> ALT
  Pathway: ROS -> snowpack ice + latent heat -> winter regime
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist']
  Confounders (explicit): ['P_annual', 'T_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'continentality_dist', 'dist_to_coast_km']

PANEL (fe_cluster): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist'] | Explicit: ['P_annual', 'T_winter_mean'] | N=296, Sites=19
  Effect: 0.5818   SE: 0.7373
  95% CI: [-0.8632, 2.0269]  p=4.3004e-01
  Std. Effect: 0.9442 cm/1-SD  (SD_treatment=1.6228)

PANEL (first_diff): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentalit

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================
import re
import os

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_DiscontinuousPermafrost.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_DiscontinuousPermafrost'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*)', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*)', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4


Loaded 390 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 390, Sites: 23
  Years: 1969-2024
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

PANEL (fe_cluster): TDD -> Max
  FE-ab

  Low: 0.0541 (-0.0273, 0.1356)   n=72
  Medium: 0.0023 (-0.0112, 0.0158)   n=144

######################################################################
# TREATMENT: FDD
######################################################################

CAUSAL ASSUMPTIONS: FDD -> ALT
  Pathway: FDD -> (snow modulates) -> soil_T_winter -> permafrost
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): ['wind_direction_winter', 'wind_speed_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'continentality_dist', 'soc_gkg_0cm']

PANEL (fe_cluster): FDD -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist', 'slope_deg', 'aspect_deg'] | Explicit: ['wind_direction_winter', 'wind_speed_winter_mean'] | N=242, Sites=17
  Effect: -0.0049   SE: 0.0027
  95% CI: [-0.0102, 0.0004]  p=7.1578e-02
  Std. Effect: -1.8098 cm/1-SD  (SD_treatment=369.0688)

PANEL (fi

  Low: 0.0067 (-0.0242, 0.0375)   n=72
  Medium: -0.0072 (-0.0139, -0.0005) *  n=144

######################################################################
# TREATMENT: SWE_max
######################################################################

CAUSAL ASSUMPTIONS: SWE_max -> ALT
  Pathway: SWE -> winter insulation -> soil_T_winter -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020']
  Confounders (explicit): ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD']
  Mediators (no control): ['snow_depth_winter', 'snow_off_doy']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'tpi']

PANEL (fe_cluster): SWE_max -> Max
  FE-absorbed: ['lat', 'elevation_m', 'slope_deg', 'aspect_deg', 'tpi', 'landformType', 'landcover_esa_2020'] | Explicit: ['P_annual', 'wind_direction_winter', 'wind_speed_winter_mean', 'LAI_summer', 'MAAT', 'FDD'] | N=242, Sites=17


  Discontinuous: -12.9187 (-97.3934, 71.5560)   n=242

--- Heterogeneity by groundIceType ---
  Low: 88.7411 (-64.6685, 242.1508)   n=72
  Medium: 34.8293 (-31.5854, 101.2439)   n=144

######################################################################
# TREATMENT: snow_off_doy
######################################################################

CAUSAL ASSUMPTIONS: snow_off_doy -> ALT
  Pathway: snow_off -> thaw season -> TDD -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'aspect_deg', 'slope_deg']
  Confounders (explicit): ['MAAT', 'SWE_max']
  Mediators (no control): ['TDD']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'continentality_dist']

PANEL (fe_cluster): snow_off_doy -> Max
  FE-absorbed: ['lat', 'elevation_m', 'aspect_deg', 'slope_deg'] | Explicit: ['MAAT', 'SWE_max'] | N=242, Sites=17
  Effect: -0.2270   SE: 0.3042
  95% CI: [-0.8232, 0.3692]  p=4.5556e-01
  Std. Effect: -1.9521 cm/1-SD  (SD_treatment=8.6003)

PANEL (first_diff):

...

  Effect: 34.4145   SE: 59.7030
  95% CI: [-82.6011, 151.4302]  p=5.6433e-01
  Std. Effect: 0.6594 cm/1-SD  (SD_treatment=0.0192)

DOWHY: soil_moisture (anomaly) -> Max | n=242

--- Heterogeneity by soil_texture_0cm ---
  7: -151.8081 (-591.6213, 288.0051)   n=165
  8: 319.7459 (-154.1304, 793.6222)   n=77

--- Heterogeneity by soc_gkg_0cm ---
  Q1: -482.6508 (-1360.4756, 395.1741)   n=100
  Q3: 147.7597 (15.7550, 279.7644) *  n=123

######################################################################
# TREATMENT: NDWI_annual_mean
######################################################################

CAUSAL ASSUMPTIONS: NDWI_annual_mean -> ALT
  Pathway: NDWI -> heat capacity / evap cooling -> ALT
  Confounders (FE absorbed): ['tpi', 'slope_deg', 'landformType', 'soil_texture_0cm']
  Confounders (explicit): []
  Mediators (no control): ['soil_moisture', 'soil_T']
  Effect modifiers: ['soil_texture_0cm', 'slope_deg', 'tpi', 'landformType']

PANEL (fe_cluster): NDWI_annual_mean -> Max

  7: 82.4716 (13.6984, 151.2449) *  n=251
  8: 320.9126 (-70.7382, 712.5634)   n=93

--- Heterogeneity by slope_deg ---
  Q2: 57.2817 (33.3751, 81.1882) *  n=132
  Q1: 192.4927 (-82.4160, 467.4014)   n=131
  Q3: 305.6340 (-126.5639, 737.8318)   n=81

######################################################################
# TREATMENT: FIRMS_fire_days
######################################################################

CAUSAL ASSUMPTIONS: FIRMS_fire_days -> ALT
  Pathway: fire -> veg/organic removal -> insulation -> ALT
  Confounders (FE absorbed): ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat']
  Confounders (explicit): ['T_summer_max', 'heat_wave_days', 'TDD']
  Mediators (no control): ['NDVI_summer', 'LAI_summer', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'continentality_dist']

PANEL (fe_cluster): FIRMS_fire_days -> Max
  FE-absorbed: ['landcover_esa_2020', 'dist_to_coast_km', 'continentality_dist', 'lat'

  Discontinuous: -0.0000 (-0.0000, 0.0000)   n=185

--- Heterogeneity by groundIceType ---
  Low: 0.0000 (0.0000, 0.0000) *  n=72
  Medium: 0.0000 (0.0000, 0.0000) *  n=91

######################################################################
# TREATMENT: heat_wave_days
######################################################################

CAUSAL ASSUMPTIONS: heat_wave_days -> ALT
  Pathway: heat extremes -> thermal pulse -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'continentality_dist']
  Confounders (explicit): ['T_annual_range']
  Mediators (no control): ['soil_T', 'soil_moisture']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landcover_esa_2020', 'soc_gkg_0cm', 'bulk_density_0cm']

PANEL (fe_cluster): heat_wave_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'continentality_dist'] | Explicit: ['T_annual_range'] | N=242, Sites=17
  Effect: -0.0158   SE: 0.1206
  95% CI: [-0.2523, 0.2207]  p=8.9580e-01
  Std. Effect: -0.1384 cm/1-SD  (SD_treatment=8.7580

  Effect: 174.7752   SE: 170.0190
  95% CI: [-158.4559, 508.0064]  p=3.0396e-01
  Std. Effect: 4.6912 cm/1-SD  (SD_treatment=0.0268)

PANEL (first_diff): NDVI_summer -> Max
  FE-absorbed: ['lat', 'soil_texture_0cm', 'soc_gkg_0cm', 'landcover_esa_2020'] | Explicit: ['MAAT', 'P_summer', 'FIRMS_fire_days'] | N=185, Sites=15
  Effect: 26.3660   SE: 71.4768
  95% CI: [-113.7260, 166.4580]  p=7.1222e-01
  Std. Effect: 0.9752 cm/1-SD  (SD_treatment=0.0370)

DOWHY: NDVI_summer (anomaly) -> Max | n=185

--- Heterogeneity by permafrostType ---
  Discontinuous: 174.7752 (-158.4559, 508.0064)   n=185

--- Heterogeneity by groundIceType ---
  Low: 619.3524 (217.9333, 1020.7715) *  n=72
  Medium: -8.4866 (-56.2731, 39.3000)   n=91



######################################################################
# TREATMENT: ROS_days
######################################################################

CAUSAL ASSUMPTIONS: ROS_days -> ALT
  Pathway: ROS -> snowpack ice + latent heat -> winter regime
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist']
  Confounders (explicit): ['P_annual', 'T_winter_mean']
  Mediators (no control): []
  Effect modifiers: ['permafrostType', 'groundIceType', 'continentality_dist', 'dist_to_coast_km']

PANEL (fe_cluster): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist'] | Explicit: ['P_annual', 'T_winter_mean'] | N=242, Sites=17
  Effect: 0.4674   SE: 0.8443
  95% CI: [-1.1874, 2.1222]  p=5.7982e-01
  Std. Effect: 0.8042 cm/1-SD  (SD_treatment=1.7205)

PANEL (first_diff): ROS_days -> Max
  FE-absorbed: ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist'] | Explicit: ['P_annual', 'T_winter_m

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================
import re
import os

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_SporadicPermafrost.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_SporadicPermafrost'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*)', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*)', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 25 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 25, Sites: 1
  Years: 1999-2024
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

--- Heterogenei

In [ ]:
# =============================================================================
# MAIN BLOCK
# =============================================================================
import re
import os

if __name__ == "__main__":
    import os, json
    from datetime import datetime

    DATA_PATH = '/content/drive/MyDrive/UND/Index/ALDI_withYearlyData_augmented_UnknownPermafrost.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/UND/Index/causal_results_UnknownPermafrost'

    TREATMENTS = ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer',
                  'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days',
                  'heat_wave_days', 'NDVI_summer', 'ROS_days']

    print("=" * 70)
    print("ALT CAUSAL INFERENCE v3.4")
    print("=" * 70)

    try:
        df = pd.read_csv(DATA_PATH)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, encoding='cp1252')
    except FileNotFoundError:
        print(f"ERROR: {DATA_PATH} not found"); exit(1)

    available = [t for t in TREATMENTS if t in df.columns]
    print(f"Loaded {len(df)} obs, {len(df.columns)} cols")
    print(f"Available treatments: {available}")

    analyzer = ALTCausalAnalysis(
        data=df, outcome="Max",
        compute_anomalies_for_dowhy=DOWHY_AVAILABLE, verbose=True
    )

    results = analyzer.full_analysis(
        treatments=available,
        methods=['fe_cluster', 'first_diff'],
        run_dowhy=DOWHY_AVAILABLE,
        run_heterogeneity=True, verbose=True
    )

    summary_df = analyzer.summary_table()
    if len(summary_df) > 0:
        print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
        print(summary_df.to_string(index=False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    if len(summary_df) > 0:
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f'causal_summary_{re.search(r'causal_results_(.*)', OUTPUT_DIR).group(1)}.csv'), index=False)

    def ser(obj):
        if isinstance(obj, dict): return {k: ser(v) for k, v in obj.items()}
        if isinstance(obj, list): return [ser(v) for v in obj]
        if isinstance(obj, (np.integer, np.floating)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if pd.isna(obj): return None
        return obj

    try:
        with open(os.path.join(OUTPUT_DIR, f'causal_full_{re.search(r'causal_results_(.*)', OUTPUT_DIR).group(1)}.json'), 'w') as f:
            json.dump(ser(results), f, indent=2)
    except Exception as e:
        print(f"JSON save failed: {e}")

    print("\n" + "=" * 70 + "\nKEY FINDINGS (fe_cluster)\n" + "=" * 70)
    for t in available:
        if t in results:
            pe = results[t].get('panel_estimates', {}).get('fe_cluster', {})
            if 'effect' in pe:
                sig = "***" if pe['p_value']<.001 else "**" if pe['p_value']<.01 else "*" if pe['p_value']<.05 else ""
                d = "increases" if pe['effect'] > 0 else "decreases"
                print(f"  {t}: +1 unit {d} ALT by {abs(pe['effect']):.3f} cm {sig}")
                print(f"    CI: [{pe['ci_lower']:.3f}, {pe['ci_upper']:.3f}]")

    print(f"\nResults: {OUTPUT_DIR}")

ALT CAUSAL INFERENCE v3.4
Loaded 29 obs, 59 cols
Available treatments: ['TDD', 'FDD', 'SWE_max', 'snow_off_doy', 'P_summer', 'soil_moisture', 'NDWI_annual_mean', 'FIRMS_fire_days', 'heat_wave_days', 'NDVI_summer', 'ROS_days']
ALT Causal Analysis v3.4
  Observations: 29, Sites: 1
  Years: 1995-2023
  Primary: Site FE + clustered SEs
  DoWhy: True (anomalies=True)

######################################################################
# TREATMENT: TDD
######################################################################

CAUSAL ASSUMPTIONS: TDD -> ALT
  Pathway: TDD -> (n-factor) -> GST_TI -> Stefan eq -> ALT
  Confounders (FE absorbed): ['lat', 'elevation_m', 'dist_to_coast_km', 'continentality_dist', 'slope_deg', 'aspect_deg']
  Confounders (explicit): []
  Mediators (no control): ['soil_T', 'soil_moisture', 'NDVI_summer', 'LAI_summer', 'NDWI_annual_mean']
  Effect modifiers: ['permafrostType', 'groundIceType', 'landformType', 'soil_texture_0cm', 'soc_gkg_0cm', 'lat']

--- Heterogenei